# Jupyter Notebook for "Using NMR to predict molecular reactivity" by Muellers et al.

All code was written by Tobias D. Muellers.

# Data
Data required to run this code is located at: https://dataverse.yale.edu/dataverse/NMR2FMO


# Environment

In [ ]:
# basic packages
import os
import sys
import numpy as np
import pandas as pd
import pickle
from tqdm import tqdm
import random
import re
import time
import notebook
from joblib import Parallel, delayed

# CPU-based AI packages
import sklearn
from sklearn.base import clone
from sklearn.model_selection import train_test_split, KFold, ParameterGrid
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.ensemble import RandomForestRegressor # for shap
import shap

# GPU-based AI packages
import cupy
import cudf
import cuml
from cuml import LinearRegression as cuml_mlr
from cuml import Lasso as cuml_lasso
from cuml import KNeighborsRegressor as cuml_knn
from cuml import SVR as cuml_svr
from cuml.ensemble import RandomForestRegressor as cuml_rf
from cuml.explainer import TreeExplainer
import xgboost
from xgboost import XGBRegressor # this can be CPU or GPU based on parameters
import torch
import torch.nn as nn
import skorch
from skorch import NeuralNetRegressor
from skorch.callbacks import EarlyStopping
from skorch.dataset import ValidSplit

# for hyperparameter optimization
import optuna
from optuna.importance import get_param_importances, FanovaImportanceEvaluator

# cheminformatics packages
import rdkit
from rdkit import Chem
from rdkit.Chem import Draw, rdMolDescriptors, Descriptors, AllChem

# plotting packages
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.font_manager as fm
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import scipy
from scipy.stats import linregress
from scipy.stats import gaussian_kde
import math

# load external functions
# Add src folder to path
sys.path.append(os.path.abspath('src'))
import functions as funs

In [ ]:
# print package versions
print(f"Python version: {sys.version}")
print(f"Jupyter version: {notebook.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"scikit-learn version: {sklearn.__version__}")
print(f"SHAP version: {shap.__version__}")
print(f"CuPy version: {cupy.__version__}")
print(f"cuDF version: {cudf.__version__}")
print(f"cuML version: {cuml.__version__}")
print(f"XGBoost version: {xgboost.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"Skorch version: {skorch.__version__}")
print(f"Optuna version: {optuna.__version__}")
print(f"RDKit version: {rdkit.__version__}")
print(f"Seaborn version: {sns.__version__}")
print(f"Matplotlib version: {mpl.__version__}")
print(f"SciPy version: {scipy.__version__}")

In [ ]:
# global parameters
dpi_for_pub = 1200
random_seed = 67923 # this seed persists through all functions where a random seed is relevant

# for plots
mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = ["Nimbus Roman"] # times new roman is not available on the system where this code was ran
resolved = fm.findfont(fm.FontProperties(family=mpl.rcParams["font.sans-serif"]),
                       fallback_to_default=False)
print(resolved)
# set for matplotlib
mpl.rcParams["mathtext.fontset"] = "stix"
plt.rcParams['font.size'] = 12

# QM9 Combined

## QM9 Data Extraction

In [ ]:
# import qm9 dataset
qm9 = funs.load_qm9() # enter appropriate directory
print(f"Number of molecules in QM9 dataset: {len(qm9)}")
print(f"Shape of QM9 dataframe: {qm9.shape}")

# import qm9 nmr dataset
qm9_nmr = funs.load_qm9_nmr() # enter appropriate directory
print(f"Number of NMR entries in QM9 NMR dataset: {len(qm9_nmr)}")
print(f"Shape of QM9 NMR dataframe: {qm9_nmr.shape}")

In [ ]:
# note that not all QM9 molecules have NMR data
# Extract the numeric part after "_" in the filename column in both DataFrames
# this creates a numeric value to merge on
qm9["file_num"] = qm9["filename"].str.split("_").str[-1].str.replace(".xyz", "", regex=False)
qm9_nmr["file_num"] = qm9_nmr["filename"].str.split("_").str[-1]

In [ ]:
# remove duplicate columns from qm9_nmr before merging
qm9_nmr = qm9_nmr.drop(columns=["filename", "num_atom"])
# Inner join on "file_num"
qm9_all = pd.merge(qm9, qm9_nmr, on="file_num", how="inner")
print(f"Number of NMR entries in merged QM9 and QM9 NMR datasets: {len(qm9_all)}")
print(f"Shape of merged QM9 and QM9 NMR dataframe: {qm9_all.shape}")

## QM9 Data Featurization

In [ ]:
# removal of unused columns
# only gas-phase NMR data is used in this work
# target is HOMO-LUMO gap
qm9_feat = qm9_all.drop(columns=["filename", "dataset", "index",
                                 "rotational_a_ghz", "rotational_b_ghz", "rotational_c_ghz",
                                 "dipole_debye", "isotropic_pol_bohr3", "homo_har",
                                 "lumo_har", "elec_spat_bohr2", "zpve_har", 
                                 "u0_har", "u_har", "h_har",
                                 "g_har", "cv_har", "inchi",
                                 "inchi_relax", "smiles_relax",
                                 "ir_freqs", "nmr_CCl4", "nmr_THF",
                                "nmr_Acetone", "nmr_Methanol", "nmr_DMSO"])

In [ ]:
# get only C and H NMR signals
qm9_feat = funs.atom_select(qm9_feat, atom_col="atoms", signal_col="nmr_gas", atom_type="C")
qm9_feat = funs.atom_select(qm9_feat, atom_col="atoms", signal_col="nmr_gas", atom_type="H")

In [ ]:
# convert to ppm

# scale via subtraction from reported reference for 13C NMR
# for gas phase, TMS 13C reference is 186.9704 ppm 
# reference from https://moldis-group.github.io/qm9nmr/
tms_13c = 186.9704

# conversion is ppm(TMS) - ppm(molecule) = shift
def subtract_from_list(lst, tms):
    return [tms - float(x) for x in lst]

qm9_feat['C_ppm'] = qm9_feat['C_signals'].apply(subtract_from_list, tms=tms_13c)

# scale via slope/intercept values for 1H NMR
# from http://cheshirenmr.info/Recommendations.htm#tantilloccs
# G09 Methods
# B3LYP/6-31+G(d,p) (gas phase)
# mPW1PW91/6-311+G(2d,p) (giao, scrf)

def scale_using_slope_intercept(shifts, slope, intercept):
    return [(shift - intercept) / slope for shift in shifts]

slope = -1.0936
intercept = 31.8018

qm9_feat['H_ppm'] = qm9_feat['H_signals'].apply(scale_using_slope_intercept, slope=slope, intercept=intercept)

# remove signals columns
qm9_feat = qm9_feat.drop(columns=['C_signals', 'H_signals'])

In [ ]:
# remove molecules with fewer than 4 carbons and 4 hydrogens
# this is required to enable calculation of kurtosis

def count_atoms(atom_list): # data in this column is of the form "list"
    return atom_list.count("H"), atom_list.count("C")

# Apply the function to create new columns for counts
qm9_feat["H_count"], qm9_feat["C_count"] = zip(*qm9_feat["atoms"].apply(count_atoms))
total_mols = len(qm9_feat)

# Filter for molecules with at least 4 H and 4 C
qm9_feat = qm9_feat[(qm9_feat["H_count"] >= 4) & (qm9_feat["C_count"] >= 4)].drop(columns=["H_count", "C_count"])
print(f"A total of {total_mols - len(qm9_feat)} molecules were removed for having fewer than 4 C or 4 H atoms.")

In [ ]:
# For columns containing arrays/lists of values
def all_values_same(arr):
    arr = np.array(arr)
    return np.all(arr == arr[0])

mask_c = qm9_feat['C_ppm'].apply(all_values_same)
mask_h = qm9_feat['H_ppm'].apply(all_values_same)

rows_to_remove = (mask_c | mask_h)
num_removed = rows_to_remove.sum()

qm9_feat = qm9_feat[~rows_to_remove].reset_index(drop=True)

print(f"The total molecules removed due to having zero variance and standard deviation was {num_removed}")

In [ ]:
# calculate summary statistics for signals
qm9_feat = funs.get_summary_stats(qm9_feat, array_col="C_ppm")
qm9_feat = funs.get_summary_stats(qm9_feat, array_col="H_ppm")

# confirm no "NaN"
nan_rows = qm9_feat[qm9_feat.isna().any(axis=1)]
print(f"A total of {len(nan_rows)} rows contain NaN values after featurization.")
print(f"A total of {len(qm9_feat)} rows remain after featurization.")

max_length = qm9_feat['H_ppm'].apply(len).max()
print("Maximum array length in H_ppm:", max_length)

max_length = qm9_feat['C_ppm'].apply(len).max()
print("Maximum array length in C_ppm:", max_length)

In [ ]:
# convert hartrees to eV
hartree_to_kcalmol = 27.211386245981 # value from NIST https://physics.nist.gov/cgi-bin/cuu/Value?hrev
# only convert columns that are in hartrees
kcal_cols =['gap_har']

# Ensure columns are numeric before multiplying
for col in kcal_cols:
    qm9_feat[col] = pd.to_numeric(qm9_feat[col], errors='coerce')

# implement conversion
qm9_feat[kcal_cols] = qm9_feat[kcal_cols] * hartree_to_kcalmol

# rename columns ending in _har to _ev
qm9_feat.rename(columns={col: col.replace('_har', '_ev') for col in kcal_cols}, inplace=True)

In [ ]:
# get additional features for explanation
qm9_feat = funs.get_all_fgs(qm9_feat, smiles_col="smiles")
qm9_feat = funs.get_lipinski_descriptors(qm9_feat, smiles_col="smiles")

In [ ]:
# rename variables
qm9_feat = qm9_feat.rename(columns={
    'C_ppm_length': '13C Number of Shifts',
    'C_ppm_min': '13C Minimum Shift',
    'C_ppm_max': '13C Maximum Shift',
    'C_ppm_mean': '13C Mean Shift',
    'C_ppm_median': '13C Median Shift',
    'C_ppm_mode': '13C Mode Shift',
    'C_ppm_var': '13C Variance',
    'C_ppm_std': '13C Standard Deviation',
    'C_ppm_skewness': '13C Skewness',
    'C_ppm_kurtosis': '13C Kurtosis',
    'H_ppm_length': '1H Number of Shifts',
    'H_ppm_min': '1H Minimum Shift',
    'H_ppm_max': '1H Maximum Shift',
    'H_ppm_mean': '1H Mean Shift',
    'H_ppm_median': '1H Median Shift',
    'H_ppm_mode': '1H Mode Shift',
    'H_ppm_var': '1H Variance',
    'H_ppm_std': '1H Standard Deviation',
    'H_ppm_skewness': '1H Skewness',
    'H_ppm_kurtosis': '1H Kurtosis'
})

In [ ]:
# checkpoint
os.makedirs("data", exist_ok=True)
os.chdir("data")
qm9_feat.to_pickle("qm9_feat.pkl")
os.chdir("..")

## QM9 CV/Test Split

In [ ]:
# specify property of interest
property_of_interest = 'gap_ev'

# specify initial features
initial_features = [
    '13C Number of Shifts',
    '13C Minimum Shift',
    '13C Maximum Shift',
    '13C Mean Shift',
    '13C Median Shift',
    '13C Mode Shift',
    '13C Variance',
    '13C Standard Deviation',
    '13C Skewness',
    '13C Kurtosis',
    '1H Number of Shifts',
    '1H Minimum Shift',
    '1H Maximum Shift',
    '1H Mean Shift',
    '1H Median Shift',
    '1H Mode Shift',
    '1H Variance',
    '1H Standard Deviation',
    '1H Skewness',
    '1H Kurtosis'
]

In [ ]:
# get x and y for stratified splitting
X = qm9_feat[initial_features]
Y = qm9_feat[property_of_interest]

# also create discretized bins for stratification
number_of_bins = 100
Y_binned = pd.qcut(Y, q=number_of_bins, labels=False, duplicates='drop')

In [ ]:
# Stratified split based on HOMO-LUMO gap
# select 20% for test set (without replacement)
x_cv, x_test, y_cv, y_test = train_test_split(X, Y, test_size=0.2, random_state=random_seed, stratify=Y_binned)

print(f'CV set size: {len(x_cv)}')
print(f'Test set size: {len(x_test)}')

## QM9 Chemical Space Plots

In [ ]:
# add features to visualize the chemical space covered by the molecules
qm9_chem_space = qm9_feat.copy()

# use rdkit for common values
qm9_chem_space['mw'] = qm9_chem_space['smiles'].apply(lambda x: Descriptors.MolWt(Chem.MolFromSmiles(x)))
qm9_chem_space['crippen_logp'] = qm9_chem_space['smiles'].apply(lambda x: Chem.Crippen.MolLogP(Chem.MolFromSmiles(x)))
qm9_chem_space['crippen_molmr'] = qm9_chem_space['smiles'].apply(lambda x: Chem.Crippen.MolMR(Chem.MolFromSmiles(x)))

In [ ]:
# calc mol volume separately
# set up vector to hold outputs
total_vols = []
vdw_vols = []
surf_areas = []
tpsas = []

# establish base parameters
params = AllChem.ETKDGv3()
params.randomSeed = random_seed

for row in tqdm(qm9_chem_space.itertuples(index=True)):
    smi = row.smiles
    mol = Chem.MolFromSmiles(smi)
    mol_with_hs = Chem.AddHs(mol) # add hydrogen for volume

    embedded = False
    max_embed_tries = 3

    for attempt in range(max_embed_tries):
        mol_with_hs.RemoveAllConformers()
        status = AllChem.EmbedMolecule(mol_with_hs, params)

        if status == 0 and mol_with_hs.GetNumConformers() > 0:
            embedded = True
            break

        # relax settings for retry
        params.useRandomCoords = True
        params.randomSeed = random_seed + attempt + 1

    if not embedded:
        total_vols.append(np.nan)
        vdw_vols.append(np.nan)
        surf_areas.append(np.nan)
        tpsas.append(np.nan)
        continue

    dclv = rdMolDescriptors.DoubleCubicLatticeVolume(mol_with_hs)
    total_volume = dclv.GetVolume()
    vdw_volume = dclv.GetVDWVolume()
    polar_surface_area = rdMolDescriptors.CalcTPSA(mol_with_hs, includeSandP=True)
    surface_area = dclv.GetSurfaceArea()

    total_vols.append(total_volume)
    vdw_vols.append(vdw_volume)
    surf_areas.append(polar_surface_area)
    tpsas.append(surface_area)
    
qm9_chem_space['total_volume'] = total_vols
qm9_chem_space['vdw_volume'] = vdw_vols
qm9_chem_space['surface_area'] = surf_areas
qm9_chem_space['polar_surface_area'] = tpsas

In [ ]:
# rename variables for plotting
qm9_chem_space = qm9_chem_space.rename(columns={
    'mw': 'Molecular weight (Da)',
    'total_volume': 'Molecular volume (Å³)',
    'vdw_volume': 'Van der Waals volume (Å³)',
    'surface_area': 'Molecular surface area (Å²)',
    'polar_surface_area': 'Polar surface area (Å²)',
    'num_atom': 'Number of atoms',
    'NumHAcceptors': 'Number of hydrogen bond acceptors',
    'NumHDonors': 'Number of hydrogen bond donors',
    'NumRotatableBonds': 'Number of rotatable bonds',
    'NumHeteroatoms': 'Number of heteroatoms',
    'RingCount': 'Number of rings',
    'NumAliphaticRings': 'Number of aliphatic rings',
    'NumAromaticRings': 'Number of aromatic rings',
    'crippen_logp': 'Octanol-water partition coefficient (Crippen)',
    'crippen_molmr': 'Molar refractivity (Crippen)'
})

chem_space_vars = [
    'Molecular weight (Da)',
    'Molecular volume (Å³)',
    'Van der Waals volume (Å³)',
    'Molecular surface area (Å²)',
    'Polar surface area (Å²)',
    'Number of atoms',
    'Number of hydrogen bond acceptors',
    'Number of hydrogen bond donors',
    'Number of rotatable bonds',
    'Number of heteroatoms',
    'Number of rings',
    'Number of aliphatic rings',
    'Number of aromatic rings',
    'Octanol-water partition coefficient (Crippen)',
    'Molar refractivity (Crippen)'
]

#### figure S10

In [ ]:
# Settings
df_plot = qm9_chem_space
vars_to_plot = chem_space_vars

n_cols = 3
n_rows = math.ceil(len(vars_to_plot) / n_cols)

sns.set_style("whitegrid")
mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = ["Nimbus Roman"]
mpl.rcParams["mathtext.fontset"] = "stix"

fig = plt.figure(figsize=(5.2 * n_cols, 2.8 * n_rows), dpi=dpi_for_pub)
outer = fig.add_gridspec(n_rows, n_cols, wspace=0.30, hspace=0.50)

for i, var in enumerate(vars_to_plot):
    r, c = divmod(i, n_cols)

    # Each panel has 2 stacked axes: top boxplot, bottom density
    inner = outer[r, c].subgridspec(2, 1, height_ratios=[1, 4], hspace=0.05)
    ax_box = fig.add_subplot(inner[0])
    ax_den = fig.add_subplot(inner[1], sharex=ax_box)

    x = pd.to_numeric(df_plot[var], errors="coerce").dropna()

    if len(x) == 0:
        ax_den.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax_den.transAxes)        
        ax_box.axis("off")
        continue

    # Top: horizontal box-and-whisker
    sns.boxplot(
        x=x,
        ax=ax_box,
        orient="h",
        color="#c7d8f5",
        fliersize=2,
        linewidth=1
    )
    ax_box.set(yticks=[], ylabel="", xlabel="")
    ax_box.tick_params(axis="x", labelbottom=False)
    ax_box.set_title("")
    for spine in ["left", "right", "top"]:
        ax_box.spines[spine].set_visible(False)

    # Bottom: density plot
    # KDE needs more than one unique value; fallback to histogram if constant
    if x.nunique() > 1:
        sns.kdeplot(x=x, ax=ax_den, fill=True, color="#2a6fbb", linewidth=1.4)
    else:
        sns.histplot(x=x, ax=ax_den, bins=15, color="#2a6fbb")

    ax_den.set_ylabel("Density")
    ax_den.set_xlabel(var)

# Turn off any unused panels
total_slots = n_rows * n_cols
for j in range(len(vars_to_plot), total_slots):
    r, c = divmod(j, n_cols)
    inner = outer[r, c].subgridspec(2, 1, height_ratios=[1, 4], hspace=0.05)
    fig.add_subplot(inner[0]).axis("off")
    fig.add_subplot(inner[1]).axis("off")

plt.tight_layout()
plots_dir = os.path.join(os.getcwd(), "plots")
os.makedirs(plots_dir, exist_ok=True)
plt.savefig(os.path.join(plots_dir, "figure s1.png"), dpi=dpi_for_pub, bbox_inches='tight')
plt.show()

## QM9 Feature Selection

In [ ]:
# min-max scaling of features, only implemented on cv set to avoid data leakage
scaler = MinMaxScaler()
min_max_array = scaler.fit_transform(x_cv)

x_cv_min_max = pd.DataFrame(min_max_array, columns=initial_features, index=x_cv.index)

# compute variance of scaled features 
variances = x_cv_min_max.var()

# Create a DataFrame with column names and variances
variance_table = pd.DataFrame({
    'column': variances.index,
    'variance': variances.values
})

# Sort by variance descending
variance_table = variance_table.sort_values(by='variance', ascending=False).reset_index(drop=True)

print(variance_table.tail(10))

In [ ]:
# drop features with variance below 0.015
low_variance_cols = variance_table[variance_table['variance'] < 0.015]['column'].tolist()
low_variance_cols_13C = [col for col in low_variance_cols if "13C" in col]
print(f"Dropping low variance columns: {low_variance_cols}")

x_cv = x_cv.drop(columns=low_variance_cols)
x_test = x_test.drop(columns=low_variance_cols)

#### figure S1

In [ ]:
# updated features
variance_filtered_features = [f for f in initial_features if f not in low_variance_cols]

# Compute correlation matrix
corr = x_cv[variance_filtered_features].corr(method='pearson')

# Create annotation matrix with asterisks for |corr| >= 0.9 and '<' for corr <= -0.9 (excluding diagonal)
def annotate_corr(x):
    if x == 1:
        return ''
    elif x >= 0.85:
        return '*'
    elif x <= -0.85:
        return '<'
    else:
        return ''
annot = corr.map(annotate_corr)

# Plot correlation heatmap
plt.figure(figsize=(12, 10), dpi=dpi_for_pub)


# Create a mapping from old variable names to new names
rename_dict = {
    '13C Number of Shifts': '$_{num}$$\\delta$$^{13C}$',
    '13C Maximum Shift': '$_{max}$$\\delta$$^{13C}$',
    '13C Mean Shift': '$_{mean}$$\\delta$$^{13C}$',
    '13C Median Shift': '$_{median}$$\\delta$$^{13C}$',
    '13C Variance': '$_{var}$$\\delta$$^{13C}$',
    '13C Standard Deviation': '$_{std}$$\\delta$$^{13C}$',
    '13C Skewness': '$_{skew}$$\\delta$$^{13C}$',
    '13C Kurtosis': '$_{kurt}$$\\delta$$^{13C}$',
    '1H Number of Shifts': '$_{num}$$\\delta$$^{1H}$',
    '1H Maximum Shift': '$_{max}$$\\delta$$^{1H}$',
    '1H Mean Shift': '$_{mean}$$\\delta$$^{1H}$',
    '1H Median Shift': '$_{median}$$\\delta$$^{1H}$',
    '1H Variance': '$_{var}$$\\delta$$^{1H}$',
    '1H Standard Deviation': '$_{std}$$\\delta$$^{1H}$',
    '1H Skewness': '$_{skew}$$\\delta$$^{1H}$',
    '1H Kurtosis': '$_{kurt}$$\\delta$$^{1H}$',
}

corr_renamed = corr.rename(index=rename_dict, columns=rename_dict)
ax = sns.heatmap(corr_renamed, annot=annot, fmt='', cmap='coolwarm', center=0, cbar_kws={'label': 'Pearson Correlation Coefficient'})

plt.tight_layout()
plt.savefig(os.path.join(plots_dir, "figure s1.png"), dpi=dpi_for_pub, bbox_inches='tight')
plt.show()

In [ ]:
# drop one of each pair of highly correlated features
high_corr_cols = ['13C Standard Deviation', '1H Median Shift',
                  '13C Variance', '13C Median Shift', '1H Mean Shift', '1H Standard Deviation'] # temp others
high_corr_cols_13C = [col for col in high_corr_cols if "13C" in col]

x_cv = x_cv.drop(columns=high_corr_cols)
x_test = x_test.drop(columns=high_corr_cols)

features = [f for f in variance_filtered_features if f not in high_corr_cols]
print(f'A total of {len(features)} features remain after variance and correlation filtering.')
print(f'The QM9 training set size to feature number ratio is {len(x_cv) / len(features):.2f}.')

In [ ]:
# Number of zero values in all data
zero_count_cv = (x_cv == 0).sum().sum()
print("Number of zero values in x_cv:", zero_count_cv)

zero_count_test = (x_test == 0).sum().sum()
print("Number of zero values in x_test:", zero_count_test)

zero_percentage = ((zero_count_cv + zero_count_test) / (x_cv.size + x_test.size)) * 100
print(f"Percentage of zero values in x_cv and x_test: {zero_percentage:.3f}%")

#### figure S2

In [ ]:
# updated features
final_features = [f for f in features if f not in low_variance_cols]

# Compute correlation matrix
corr = x_cv[final_features].corr(method='pearson')

# Create annotation matrix with asterisks for |corr| >= 0.9 and '<' for corr <= -0.9 (excluding diagonal)
def annotate_corr(x):
    if x == 1:
        return ''
    elif x >= 0.85:
        return '*'
    elif x <= -0.85:
        return '<'
    else:
        return ''
annot = corr.map(annotate_corr)

# Plot correlation heatmap
plt.figure(figsize=(12, 10), dpi=dpi_for_pub)

# Create a mapping from old variable names to new names
rename_dict = {
    '13C Number of Shifts': '$_{num}$$\\delta$$^{13C}$',
    '13C Maximum Shift': '$_{max}$$\\delta$$^{13C}$',
    '13C Mean Shift': '$_{mean}$$\\delta$$^{13C}$',
    '13C Skewness': '$_{skew}$$\\delta$$^{13C}$',
    '13C Kurtosis': '$_{kurt}$$\\delta$$^{13C}$',
    '1H Number of Shifts': '$_{num}$$\\delta$$^{1H}$',
    '1H Maximum Shift': '$_{max}$$\\delta$$^{1H}$',
    '1H Mean Shift': '$_{mean}$$\\delta$$^{1H}$',
    '1H Skewness': '$_{skew}$$\\delta$$^{1H}$'
}

corr_renamed = corr.rename(index=rename_dict, columns=rename_dict)
ax = sns.heatmap(corr_renamed, annot=annot, fmt='', cmap='coolwarm', center=0, cbar_kws={'label': 'Pearson Correlation Coefficient'})

plt.tight_layout()
plots_dir = os.path.join(os.getcwd(), "plots")
plt.savefig(os.path.join(plots_dir, "figure s2.png"), dpi=dpi_for_pub, bbox_inches='tight')
plt.show()

In [ ]:
# checkpoint
os.chdir("data")
x_cv.to_pickle("x_cv.pkl")
y_cv.to_pickle("y_cv.pkl")
x_test.to_pickle("x_test.pkl")
y_test.to_pickle("y_test.pkl")
with open('features.pkl', 'wb') as f:
    pickle.dump(features, f)
os.chdir("..")

## QM9 Hyperparameter Optimization

In [ ]:
# Check if cuML is using the GPU
try:
    n_devices = cupy.cuda.runtime.getDeviceCount()
    current_device = cupy.cuda.Device()
    print(f"Number of CUDA devices detected by CuPy: {n_devices}")
    print(f"Current CUDA device: {current_device}")
except Exception as e:
    print(f"Error checking cuML/CuPy GPU usage: {e}")

In [ ]:
# Build simple mlp
def set_global_seed(seed: int = random_seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    # torch.backends.cudnn.deterministic = True T not used due to slowdown of MLP
    torch.backends.cudnn.benchmark = True

# specify possible activation functions
_ACTS = {
    "relu": nn.ReLU,
    "elu": nn.ELU,
    "gelu": nn.GELU,
    "leaky_relu": nn.LeakyReLU,
    "tanh": nn.Tanh,
    "sigmoid": nn.Sigmoid
}

class MLPRegressorModule(nn.Module):
    def __init__(
        self,
        input_dim: int,
        hidden_layer_sizes=(8, 6, 4), # inital layer sizes
        activation="relu", # base case with relu
        dropout=0.0, # no dropout for base
    ):
        super().__init__()
        if activation not in _ACTS:
            raise ValueError(f"activation must be one of {list(_ACTS.keys())}")

        act = _ACTS[activation]
        h1, h2, h3 = hidden_layer_sizes

        self.net = nn.Sequential(
            nn.Linear(input_dim, h1),
            act(),
            nn.Dropout(dropout),

            nn.Linear(h1, h2),
            act(),
            nn.Dropout(dropout),

            nn.Linear(h2, h3),
            act(),
            nn.Dropout(dropout),

            nn.Linear(h3, 1),
        )

    def forward(self, X):
        return self.net(X)

def make_mlp_skorch_estimator(input_dim: int):
    net = NeuralNetRegressor(
        module=MLPRegressorModule,
        module__input_dim=input_dim,
        module__hidden_layer_sizes=(8, 6, 4),
        module__activation="relu",
        module__dropout=0.0,

        criterion=nn.MSELoss, # only training by MSE
        optimizer=torch.optim.Adam, # only using Adam

        lr=1e-3,
        max_epochs=100,
        batch_size=4096,

        device="cuda" if torch.cuda.is_available() else "cpu",

        # we're enabling validation splits inside so we can aggressively monitor validation loss
        train_split=ValidSplit(cv=0.1, stratified=False, random_state=random_seed),

        callbacks=[
            EarlyStopping(monitor="valid_loss", patience=8, threshold=1e-3, lower_is_better=True),
        ],

        verbose=0,
    )
    return net

In [ ]:
# set up k-fold cv to use inside Optuna
cv = 10 # 10-fold cross-validation
kf = KFold(n_splits=cv, shuffle=True, random_state=random_seed)

In [ ]:
# create precomputed folds to speed up loop
precomputed_folds = [
    (train_idx.astype(np.int32), test_idx.astype(np.int32))
    for train_idx, test_idx in kf.split(x_cv)
]
print(f"Precomputed {len(precomputed_folds)} folds")

In [ ]:
# List of dictionaries of models for hyperparameter tuning
gpu_models_and_params = [
    {'name': 'MLR', 'estimator': cuml_mlr()},
    {'name': 'Lasso', 'estimator': cuml_lasso(
        max_iter = 20000
    )},
    {'name': 'KNN', 'estimator': cuml_knn()},
    {'name': 'SVR', 'estimator': cuml_svr()},
    {'name': 'RandomForest', 'estimator': cuml_rf(
        random_state = random_seed
    )},
    {'name': 'XGBoost', 'estimator': XGBRegressor(
        random_state = random_seed,
        tree_method = 'hist',
        device = 'cuda'
    )},
    {'name': 'MLP_skorch', 'estimator': make_mlp_skorch_estimator(input_dim=x_cv.shape[1])}
]

In [ ]:
# implement hpo search using optuna
# here we use different studies per model family so that we have the same number of trials for each model family

def suggest_params(trial, model_name, input_dim=None):
    # Return a dict of params to pass into model.set_params(**params)
    if model_name == "MLR":
        return {}
        
    if model_name == "Lasso":
        return {
            'alpha': trial.suggest_float('alpha', 1e-5, 1e1, log=True),
            'tol': trial.suggest_float('tol', 1e-6, 1e-2, log=True)
    }

    if model_name == "KNN":
        return {
            'n_neighbors': trial.suggest_int('n_neighbors', 3, 200, log=True),
            'metric': trial.suggest_categorical('metric', ['euclidean', 'manhattan']),
            'weights': trial.suggest_categorical('weights', ['uniform', 'distance'])
        }

    if model_name == 'SVR':
        return {
            'max_iter': trial.suggest_int('max_iter', 200, 5000, log=True),
            'kernel': trial.suggest_categorical('kernel', ['linear', 'rbf']),
            'C': trial.suggest_float('C', 1e-2, 1e3, log=True),
            'epsilon': trial.suggest_float('epsilon', 1e-3, 1.0, log=True),
            'gamma': trial.suggest_float('gamma', 1e-4, 1e1, log=True) # note that this only applies to the rbf kernel
        }
    if model_name == 'RandomForest':
        return {
            'n_estimators': trial.suggest_int('n_estimators', 100, 1200),
            'split_criterion': trial.suggest_categorical('split_criterion', ['mse', 'poisson', 'gamma', 'inverse_gaussian']), # all but mse have requirements for y target values
            'max_depth': trial.suggest_int('max_depth', 3, 30),
            'max_features': trial.suggest_categorical('max_features', ['log2', 'sqrt', 0.3, 0.5, 1.0]),
            'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20),
            'min_samples_split': trial.suggest_int('min_samples_split', 2, 50),
            'bootstrap': trial.suggest_categorical('bootstrap', [True, False])
        }
    if model_name == 'XGBoost':
        return {
            'max_depth':  trial.suggest_int('max_depth', 2, 10),
            'eta': trial.suggest_float('eta', 1e-2, 3e-1, log=True),
            'n_estimators': trial.suggest_int('n_estimators', 200, 3000),
            'subsample': trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'min_child_weight': trial.suggest_float('min_child_weight', 1e-2, 20.0, log=True),
            'gamma': trial.suggest_float('gamma', 1e-4, 5.0, log=True),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 1.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 100.0, log=True),
        }
    if model_name == "MLP_skorch":
        seed = trial.suggest_categorical('seed', [random_seed]) # pin to random seed
        h1 = trial.suggest_int("h1", 16, 64, step=8)
        h2 = trial.suggest_int("h2", 8, 32, step=4)
        h3 = trial.suggest_int("h3", 4, 16, step=2)

        return {
            # skorch top-level params
            "seed": seed,
            "device": trial.suggest_categorical("device", ["cuda"]),  # only run on GPU
            "max_epochs": trial.suggest_int("max_epochs", 50, 225, step=25),
            "batch_size": trial.suggest_categorical("batch_size", [1024, 2048, 4096, 8192, 16384]),
            "lr": trial.suggest_float("lr", 5e-4, 2e-1, log=True), # this samples from log distribution

            # optimizer params
            "optimizer__weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True),

            # module params
            "module__input_dim": input_dim,
            "module__hidden_layer_sizes": (h1, h2, h3),
            "module__activation": trial.suggest_categorical("activation", ["relu", "elu", "gelu", "leaky_relu", "tanh", "sigmoid"]),
            "module__dropout": trial.suggest_float("dropout", 1e-6, 0.5),

            "verbose": 0,
        }
        
    return {}

In [ ]:
def make_objective(model_name, base_estimator, x_cv, y_cv, folds, results_store, property_name="HOMO-LUMO Gap"):
    def objective(trial):
        # make a "base/default" evaluation as trial 0        
        if trial.number == 0:
            params = {}
        else:
            params = suggest_params(trial, model_name, input_dim=x_cv.shape[1])
        seed = None
        if model_name == "MLP_skorch" and params and "seed" in params:
            params = dict(params)
            seed = int(params.pop("seed"))
            
        model = clone(base_estimator)
        if params:
            model.set_params(**params)

        run_label = f"{model_name}__base" if not params else f"{model_name}__{params}"
        if model_name == "MLP_skorch" and seed is not None:
            run_label += f"__seed={seed}"
                
        mse_folds, rmse_folds, mae_folds, q2_folds = [], [], [], []

        for fold_idx, (train_idx, test_idx) in enumerate(folds, start=1):
            x_tr, x_te = x_cv.iloc[train_idx], x_cv.iloc[test_idx]
            y_tr, y_te = y_cv.iloc[train_idx], y_cv.iloc[test_idx]

            # tree-based not need scaling
            if model_name in ("RandomForest", "XGBoost"):
                x_tr_norm, x_te_norm = x_tr, x_te
            else: # Standard scaling on train applied to test
                scaler = StandardScaler()
                x_tr_norm = scaler.fit_transform(x_tr)
                x_te_norm = scaler.transform(x_te)
                
            try:
                if model_name == "XGBoost":
                    x_tr_gpu = cudf.from_pandas(x_tr_norm)
                    x_te_gpu = cudf.from_pandas(x_te_norm)
                    y_tr_gpu = cudf.Series(y_tr.values)
                    model.fit(x_tr_gpu, y_tr_gpu)
                    y_pred = model.predict(x_te_gpu)
                elif model_name == "MLP_skorch":
                    if seed is not None:
                        set_global_seed(int(random_seed))

                    # skorch expects numpy float32; y typically as (N, 1)
                    x_tr32 = x_tr_norm.astype(np.float32, copy=False)
                    x_te32 = x_te_norm.astype(np.float32, copy=False)
                    y_tr32 = y_tr.to_numpy(dtype=np.float32).reshape(-1, 1)

                    t0 = time.time()
                    model.fit(x_tr32, y_tr32)
                    device_used = next(model.module_.parameters()).device
                    n_epochs_run = len(model.history)
                    print(f"[{device_used}] fit took {time.time() - t0:.1f}s over {n_epochs_run} epochs")
                    assert device_used.type == "cuda", "MLP fell back to CPU!"
                    y_pred = model.predict(x_te32).reshape(-1)
                else:
                    model.fit(x_tr_norm, y_tr)
                    y_pred = model.predict(x_te_norm)

                # metrics for saving and pruning
                fold_mse = mean_squared_error(y_te, y_pred)
                mse_folds.append(mean_squared_error(y_te, y_pred))
                rmse_folds.append(root_mean_squared_error(y_te, y_pred))
                mae_folds.append(mean_absolute_error(y_te, y_pred))
                q2_folds.append(r2_score(y_te, y_pred))

                # pruning
                running_mse = float(np.mean(mse_folds))
                trial.report(running_mse, step=fold_idx)  # step == CV fold count

                if trial.should_prune():
                    raise optuna.exceptions.TrialPruned()

            except optuna.exceptions.TrialPruned:
                # IMPORTANT: don't swallow pruning
                raise
            except Exception as e:
                print(f"Fold error for {run_label}: {e}")
                continue

        # If all folds failed, penalize the trial
        if len(rmse_folds) == 0:
            return float("inf")

        row = {
            "Property": property_name,
            "Model": run_label,
            "trial_number": trial.number,
            "MSE_mean": float(np.mean(mse_folds)),
            "MSE_std": float(np.std(mse_folds)),
            "RMSE_mean": float(np.mean(rmse_folds)),
            "RMSE_std": float(np.std(rmse_folds)),
            "MAE_mean": float(np.mean(mae_folds)),
            "MAE_std": float(np.std(mae_folds)),
            "Q2_mean": float(np.mean(q2_folds)),
            "Q2_std": float(np.std(q2_folds)),
            "n_folds_used": len(rmse_folds),
            "model_name": model_name,
        }
        
        # Persist metrics into Optuna storage (SQLite) for this trial
        for k in ["MSE_mean","MSE_std","RMSE_mean","RMSE_std","MAE_mean","MAE_std","Q2_mean","Q2_std","n_folds_used"]:
            trial.set_user_attr(k, row[k])
        
        # Optional: store a compact label / params used
        trial.set_user_attr("run_label", run_label)
        trial.set_user_attr("model_name", model_name)
        trial.set_user_attr("Property", property_name)

        # Optimize MSE (minimize)
        return row["MSE_mean"]

    return objective

In [ ]:
# Set up for persistent storage
runs_dir = os.path.join(os.getcwd(), "hpo")
os.makedirs(runs_dir, exist_ok=True)

db_path = os.path.join(runs_dir, "optuna_qm9_hpo.db")

# Note: sqlite://// is correct for an absolute Linux path.
OPTUNA_STORAGE = f"sqlite:////{db_path}"

# Optional: include property name so you don't mix studies across targets
PROPERTY_NAME = "HOMO-LUMO Gap"
STUDY_PREFIX = f"MSE__{PROPERTY_NAME}"

# add a pruner
pruner = optuna.pruners.MedianPruner(
    n_startup_trials=20,   # don’t prune the first few trials
    n_warmup_steps=4,     # don’t prune before cv fold 4
    interval_steps=1
)

def get_or_create_study(model_name: str):
    return optuna.create_study(
        direction="minimize",
        study_name=f"{STUDY_PREFIX}__{model_name}",
        storage=OPTUNA_STORAGE,
        load_if_exists=True,
        pruner=pruner # adds pruning
    )

# Build/load all studies once
studies = {m["name"]: get_or_create_study(m["name"]) for m in gpu_models_and_params}

# Quick check
for name, st in studies.items():
    print(name, "existing trials:", len(st.trials))

In [ ]:
results = []  # if you want *results across this session*
N_TRIALS = 200

def run_hpo_for_model(model_name: str, n_trials: int):
    model_spec = next(m for m in gpu_models_and_params if m["name"] == model_name)
    base_estimator = model_spec["estimator"]

    study = studies[model_name]  # already persistent

    objective = make_objective(
        model_name=model_name,
        base_estimator=base_estimator,
        x_cv=x_cv,
        y_cv=y_cv,
        folds=precomputed_folds,
        results_store=results,
        property_name=PROPERTY_NAME
    )

    study.optimize(objective, n_trials=n_trials)
    return study

# Run all models (like your current loop), but now resumable:
for model_spec in tqdm(gpu_models_and_params, desc="Model loop"):
    model_name = model_spec["name"]
    n_trials = 1 if model_name == "MLR" else N_TRIALS
    run_hpo_for_model(model_name, n_trials=n_trials)

In [ ]:
def best_params_str(study):
    bp = study.best_trial.params
    return "base/default" if len(bp) == 0 else ", ".join(f"{k}={v}" for k, v in bp.items())

rows = []
for m, st in studies.items():
    bt = st.best_trial
    ua = bt.user_attrs

    rows.append({
        "model_name": m,
        "MSE_mean": ua.get("MSE_mean", float(bt.value)),
        "MSE_std": ua.get("MSE_std", np.nan),
        "RMSE_mean": ua.get("RMSE_mean", np.nan),
        "RMSE_std": ua.get("RMSE_std", np.nan),
        "MAE_mean": ua.get("MAE_mean", np.nan),
        "MAE_std": ua.get("MAE_std", np.nan),
        "Q2_mean": ua.get("Q2_mean", np.nan),
        "Q2_std": ua.get("Q2_std", np.nan),
        "n_folds_used": ua.get("n_folds_used", np.nan),
        "best_params_str": best_params_str(st),
        "best_trial_number": bt.number,
        "n_trials_total": len(st.trials),
    })

qm9_leaderboard = pd.DataFrame(rows).sort_values("MSE_mean").reset_index(drop=True)
qm9_leaderboard

In [ ]:
# Save HPO results table to CSV
os.chdir("hpo")
qm9_leaderboard.to_csv("qm9_leaderboard.csv", index=False)
print("Saved QM9 HPO results.csv")
os.chdir("..")

## QM9 Hyperparameter Influence

In [ ]:
def _non_constant_params(study: optuna.study.Study):
    complete = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    params = sorted({k for t in complete for k in t.params.keys()})
    keep, dropped = [], []
    for p in params:
        vals = [t.params[p] for t in complete if p in t.params]
        (keep if len(set(vals)) >= 2 else dropped).append(p)
    return keep, dropped
    
def plot_global_importances_2perrow(
    studies,                 # dict: {model_name: study}
    top_n: int | None = 15,
    figsize=(8, 4),          # per-plot size (w, h)
    drop_constants: bool = True,
    evaluator=None,
    show_dropped: bool = True,
    dpi=None,                # e.g. dpi_for_pub
    skip_names=("MLR",),
):
    items = [(name, st) for name, st in studies.items() if name not in set(skip_names)]
    n = len(items)
    ncols = 2
    nrows = math.ceil(n / ncols)

    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(figsize[0] * ncols, figsize[1] * nrows),
        dpi=dpi,
        squeeze=False
    )

    for i, (model_name, study) in enumerate(items):
        r, c = divmod(i, ncols)
        ax = axes[r, c]

        # --- BEGIN: your existing code, with ONLY "ax=" substitutions and no plt.show() ---
        complete = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
        if not any(t.params for t in complete):
            ax.axis("off")
            ax.text(0.0, 0.6, f"{model_name} ({len(study.trials)} trials)\n(no COMPLETE trials with params)", fontsize=12)
            continue

        if evaluator is None:
            _evaluator = FanovaImportanceEvaluator(seed=0)
        else:
            _evaluator = evaluator

        if drop_constants:
            params_keep, params_dropped = _non_constant_params(study)
        else:
            params_keep = None
            params_dropped = []

        try:
            imp = get_param_importances(study, evaluator=_evaluator, params=params_keep)
        except Exception as e:
            ax.axis("off")
            ax.text(
                0.0, 0.6,
                f"{model_name} ({len(study.trials)} trials)\n"
                f"importance computation failed:\n{type(e).__name__}: {e}",
                fontsize=11
            )
            continue

        if not imp:
            ax.axis("off")
            msg = f"{model_name} ({len(study.trials)} trials)\n(no importances returned)"
            if drop_constants and show_dropped and params_dropped:
                msg += f"\n(dropped constants: {', '.join(params_dropped)})"
            ax.text(0.0, 0.6, msg, fontsize=11)
            continue

        items2 = list(imp.items())
        if top_n is not None:
            items2 = items2[:top_n]

        labels = [k for k, _ in items2][::-1]
        vals = [v for _, v in items2][::-1]

        ax.barh(labels, vals, color="0.6")
        ax.set_xlabel("Importance")
        ax.set_title(f"{model_name} (200 trials) — Global hyperparameter importances") # three MLP failures means there were technically 103 MLP trials
        ax.set_xlim(0, max(vals) * 1.1)

        if drop_constants and show_dropped and params_dropped:
            ax.text(
                1.0, -0.12,
                f"Dropped constant params: {', '.join(params_dropped)}",
                transform=ax.transAxes,
                ha="right", va="top",
                fontsize=9
            )
        # --- END: your existing code ---

    # turn off any unused subplot(s)
    for j in range(n, nrows * ncols):
        r, c = divmod(j, ncols)
        axes[r, c].axis("off")

    plt.tight_layout()
    plots_dir = os.path.join(os.getcwd(), "plots")
    plt.savefig(os.path.join(plots_dir, "figure s12.png"), dpi=dpi_for_pub, bbox_inches='tight')
    plt.show()

In [ ]:
plot_global_importances_2perrow(
    studies,
    top_n=10,
    figsize=(8, 4),
    dpi=dpi_for_pub,
)

## QM9 Evaluation

In [ ]:
# starting with base models,
base_models = {
    "MLR": cuml_mlr(),
    "Lasso": cuml_lasso(),
    "KNN": cuml_knn(),
    "SVR": cuml_svr(),
    "RandomForest": cuml_rf(),
    "RandomForest_cpu": RandomForestRegressor(
    n_estimators=647,
    criterion='squared_error',   # cuML's 'mse' maps to sklearn's 'squared_error'
    max_depth=24,
    max_features=0.5,
    min_samples_leaf=1,
    min_samples_split=3,
    bootstrap=False,
    random_state=random_seed,    # match your pinned seed for reproducibility
    n_jobs=-1,                   # use all CPU cores; sklearn RF doesn't parallelize by default
    ),
    "XGBoost": XGBRegressor(),
    "MLP_skorch": make_mlp_skorch_estimator(input_dim=x_cv.shape[1])
}

# create final fitted-ready estimators using best params from Optuna DB
final_models = {}
for name, base_est in base_models.items():
    est = clone(base_est)
    
    if name == "SVR":
        # Best trial was the base/default config — use estimator defaults as-is
        final_models[name] = est
        continue
    if name == "RandomForest_cpu":
        final_models[name] = est
        continue
        
    raw_best = studies[name].best_trial.params

    # Replay the same suggest_params logic with fixed values,
    # so h1/h2/h3 -> hidden_layer_sizes, weight_decay -> optimizer__weight_decay,
    # module__input_dim gets included, etc. all happen automatically.
    fixed_trial = optuna.trial.FixedTrial(raw_best)
    best_params = suggest_params(fixed_trial, name, input_dim=x_cv.shape[1])

    # seed isn't a real skorch/estimator param, still needs dropping
    best_params = {k: v for k, v in best_params.items() if k not in {"seed"}}

    if best_params:
        est.set_params(**best_params)
    final_models[name] = est

final_models_and_params = [{"name": k, "estimator": v} for k, v in final_models.items()]

In [ ]:
# train and test best models
results = []

# Z-score normalization (fit only on training set)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(x_cv).astype(np.float32)
X_test_scaled = scaler.transform(x_test).astype(np.float32)

for model_info in tqdm(final_models_and_params):
    name = model_info['name']
    estimator = model_info['estimator']

    if name == "MLP_skorch":
        y_cv = y_cv.to_numpy(dtype=np.float32).reshape(-1, 1)
    else:
        y_cv = y_cv
        
    # Fit on training set
    estimator.fit(X_train_scaled, y_cv)
    # Predict on test set
    y_pred = estimator.predict(X_test_scaled)
    if name == "MLP_skorch":
        y_pred = y_pred.reshape(-1)
    # Metrics
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    results.append({
        'Model': name,
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2
        
    })

test_results_df = pd.DataFrame(results)

In [ ]:
test_results_df.head(8)

In [ ]:
# train best model on full cv set for plotting

# gpu based, but this doesn't work for SHAP given model size
qm9_estimator_gpu = next(d["estimator"] for d in final_models_and_params if d["name"] == "RandomForest")

# cpu based, which is better for shap
qm9_estimator_cpu = RandomForestRegressor(
    n_estimators=647,
    criterion='squared_error',   # cuML's 'mse' maps to sklearn's 'squared_error'
    max_depth=24,
    max_features=0.5,
    min_samples_leaf=1,
    min_samples_split=3,
    bootstrap=False,
    random_state=random_seed,    # match your pinned seed for reproducibility
    n_jobs=-1,                   # use all CPU cores; sklearn RF doesn't parallelize by default
)

# gpu vs cpu selection
qm9_estimator = qm9_estimator_gpu

results = []

# Z-score normalization (fit only on training set)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(x_cv)
X_test_scaled = scaler.transform(x_test)

# Fit on training set
qm9_estimator.fit(X_train_scaled, y_cv)

# Predict on test set
y_pred = qm9_estimator.predict(X_test_scaled)

# Metrics
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

# store predictions and residuals for plotting
residuals = y_test - y_pred
xgb_df = pd.DataFrame({
    'y_true': y_test,
    'y_pred': y_pred,
    'residual': residuals
})

In [ ]:
# recover index based on x_test to create results for plotting
qm9_test = qm9_feat.loc[x_test.index].copy()

# selected model results for plotting
selected_model_df = pd.concat([xgb_df.reset_index(drop=True), qm9_test.reset_index(drop=True)], axis=1)

## QM9 Plotting

In [ ]:
# add explanatory features
selected_model_df['zwitterion'] = selected_model_df['smiles'].apply(lambda x: int('+' in x and '-' in x))
selected_model_df['charged'] = selected_model_df['smiles'].apply(lambda x: int('+' in x or '-' in x))

### figure 2

In [ ]:
residuals_all_abs = np.abs(selected_model_df['residual'].values)
fig = plt.figure(figsize=(10, 6), dpi=dpi_for_pub)
gs = gridspec.GridSpec(2, 2, width_ratios=[2.5, 1], height_ratios=[1, 1])

# Panel A: predicted vs actual
ax0 = fig.add_subplot(gs[:, 0])
x = selected_model_df['y_true'].values
y = selected_model_df['y_pred'].values
is_zwitterion = selected_model_df['zwitterion'].astype(bool).values
#is_halogen = selected_model_df['fr_halogen'].astype(float).values > 1
xy = np.vstack([x, y])
density = gaussian_kde(xy)(xy)
ax0.scatter(x[is_zwitterion], y[is_zwitterion], facecolors='none', edgecolors='orange', s=40, linewidths=1.2, zorder=1)

#Add an arrow and label to highlight the orange-circled points (zwitterions)
arrow_start = (1.0, 8)   # (x, y) where the label will be placed
arrow_end = (3, 7)     # (x, y) where the arrow points (center of orange group)

ax0.annotate(
    'Zwitterions',
    xy=arrow_end, xycoords='data',           # Arrow points here
    xytext=arrow_start, textcoords='data',   # Label placed here
    arrowprops=dict(facecolor='orange', edgecolor='orange', arrowstyle='->', lw=2),
    fontsize=13, color='orange', fontweight='bold', ha='left', va='bottom'
)

# add metrics to Panel B lower
metrics_text_a = (
    f"R$^{2}$ = {r2:.3f}\n"
    f"MAE = {mae:.3f}\n"
    f"MSE = {mse:.3f}\n"
    f"RMSE = {rmse:.3f}"
)
ax0.text(
    0.98, 0.02, metrics_text_a,
    transform=ax0.transAxes,
    fontsize=10,
    verticalalignment='bottom',
    horizontalalignment='right',
    bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='black', alpha=0.8)
)

sc = ax0.scatter(x, y, c=density, cmap='viridis', s=8, alpha=0.8, zorder=2)
lo = min(x.min(), y.min())
hi = max(x.max(), y.max())
ax0.plot([lo, hi], [lo, hi], 'r--', linewidth=1)
ax0.set_xlabel('B3LYP/6-31G(2df,p) computed $\\Delta$E (eV)')
ax0.set_ylabel('NMR-predicted $\\Delta$E (eV)')
ax0.grid(False)
cbar = fig.colorbar(sc, ax=ax0)
cbar.set_label('Point density')
ax0.text(-0.10, 0.98, 'A', transform=ax0.transAxes, fontsize=16, fontweight='bold', va='top', ha='right')

# Panel B: signed residuals (hist + KDE)
ax1 = fig.add_subplot(gs[0, 1])
sns.histplot(selected_model_df['residual'], bins=100, kde=True, stat='density', ax=ax1, color='C0', edgecolor=None)
ax1.set_xlabel('$\\Delta$E prediction error (eV)')
ax1.set_ylabel('Density')
ax1.text(-0.20, 0.98, 'B', transform=ax1.transAxes, fontsize=16, fontweight='bold', va='top', ha='right')
residuals = selected_model_df['residual'].values
p95 = np.percentile(residuals, 95)
p5 = np.percentile(residuals, 5)
ax1.axvline(p95, color='red', linestyle='--', linewidth=2, label=f'95th percentile: {p95:.2f}')
ax1.axvline(p5, color='red', linestyle='--', linewidth=2, label=f'5th percentile: {p5:.2f}')

# Place label to the right of the 95th percentile line
ax1.text(p95 + 0.2, ax1.get_ylim()[1]*0.6, f'95th:\n{p95:.2f}', color='red', fontsize=10, ha='left', va='bottom', fontweight='normal')

# Place label to the left of the 5th percentile line
ax1.text(p5 - 0.2, ax1.get_ylim()[1]*0.6, f'5th:\n{p5:.2f}', color='red', fontsize=10, ha='right', va='bottom', fontweight='normal')

# Panel C: chemical structures of largest residuals
ax2 = fig.add_subplot(gs[1, 1])
largest_idx = residuals_all_abs.argsort()[-4:][::-1]
smiles_list = selected_model_df.iloc[largest_idx]['smiles'].tolist()
mol_list = [Chem.MolFromSmiles(smi) for smi in smiles_list]
img = Draw.MolsToGridImage(
    mol_list,
    molsPerRow=2,
    subImgSize=(150,150),
    legends=[f"Residual = {selected_model_df.iloc[i]['residual']:.2f} eV" for i in largest_idx],
    returnPNG=False
)
ax2.axis('off')
ax2.text(-0.20, 1.05, 'C', transform=ax2.transAxes, fontsize=16, fontweight='bold', va='top', ha='right')
img_rgb = img.convert("RGB")
img_array = np.array(img_rgb)
ax2.imshow(img_array)

# Save the image to the 'plots' folder
plots_dir = os.path.join(os.getcwd(), "plots")
plt.tight_layout()
fig.savefig(os.path.join(plots_dir, "figure 2.png"), dpi=dpi_for_pub, bbox_inches='tight')
plt.show(fig)

In [ ]:
# Calculate percentage of predictions with |residual| < 1.0 eV
num_below_1ev = (np.abs(selected_model_df['residual']) < 1.0).sum()
percent_below_1ev = 100 * num_below_1ev / len(selected_model_df)
print(f"Percentage of predictions with |residual| < 1.0 eV: {percent_below_1ev:.1f}%")

# compute 5th and 95th percentiles
p5 = np.percentile(selected_model_df['residual'], 5)
p95 = np.percentile(selected_model_df['residual'], 95)
print(f"5th percentile: {p5:.2f}, 95th percentile: {p95:.2f}")

In [ ]:
# compute the average absolute error for zwitterions as a separate group
mae_zwitterion = np.mean(np.abs(selected_model_df.loc[selected_model_df['zwitterion'] == 1, 'residual']))
print(f"MAE for zwitterions: {mae_zwitterion:.4f}")

# compute the average absolute error for non-zwitterions as a separate group
mae_non_zwitterion = np.mean(np.abs(selected_model_df.loc[selected_model_df['zwitterion'] == 0, 'residual']))
print(f"MAE for non-zwitterions: {mae_non_zwitterion:.4f}")

# compute the number of zwitterions
num_zwitterions = selected_model_df['zwitterion'].sum()
print(f"Number of zwitterions: {num_zwitterions}")

# compute the average absolute error for halogen-containing molecules as a separate group
mae_halogen = np.mean(np.abs(selected_model_df.loc[selected_model_df['fr_halogen'] > 1, 'residual']))
print(f"MAE for halogen-containing molecules: {mae_halogen:.4f}")

# compute the average absolute error for non-halogen-containing molecules as a separate group
mae_non_halogen = np.mean(np.abs(selected_model_df.loc[selected_model_df['fr_halogen'] <= 1, 'residual']))
print(f"MAE for non-halogen-containing molecules: {mae_non_halogen:.4f}")

### figure 3

In [ ]:
# get feature importances for best model
qm9_model_importances = qm9_estimator.feature_importances_

In [ ]:
# run on 25% of cv dataset
print(len(X_train_scaled)*0.25)

In [ ]:
# parallelize to speed up SHAP
N_ROWS = 25433
X_subset = X_train_scaled[0:N_ROWS]

def explain_chunk(model, X_chunk):
    explainer = shap.TreeExplainer(model, feature_perturbation="tree_path_dependent")
    values = explainer.shap_values(X_chunk, check_additivity=True)
    base_value = explainer.expected_value
    return values, base_value

n_jobs = 32
chunks = np.array_split(X_subset, n_jobs)

results = Parallel(n_jobs=n_jobs, backend="loky")(
    delayed(explain_chunk)(qm9_estimator, chunk) for chunk in chunks
)

# Reassemble
shap_values_list, base_values_list = zip(*results)
shap_values_full = np.concatenate(shap_values_list, axis=0)

# base_value is a scalar (or array of 1 for regression) - same across chunks, just grab one
base_value = base_values_list[0]
if isinstance(base_value, np.ndarray):
    base_value = base_value.item() if base_value.size == 1 else base_value

# Build base_values array matching number of rows (Explanation expects one per row)
base_values_full = np.full(shap_values_full.shape[0], base_value)

# Build the full Explanation object, same shape as before
qm9_explanation = shap.Explanation(
    values=shap_values_full,
    base_values=base_values_full,
    data=X_subset,
    feature_names=features
)

In [ ]:
shap_features_subset = ["13C Maximum Shift", "1H Number of Shifts"]

# Get indices of selected features
shap_indices_subset = [features.index(f) for f in shap_features_subset]

# Slice values and data for those features
shap_values_subset = qm9_explanation.values[:, shap_indices_subset]
shap_data_subset = X_train_scaled[0:N_ROWS, shap_indices_subset] # flag replace with :

explanation_subset = shap.Explanation(
    values=shap_values_subset,
    base_values=qm9_explanation.base_values,
    data=shap_data_subset,
    feature_names=shap_features_subset
)

full_explanation = shap.Explanation(
    values=qm9_explanation.values,
    base_values=qm9_explanation.base_values,
    data=X_train_scaled,
    feature_names=features
)

In [ ]:
mpl.rcParams.update({'font.size': 10})
mpl.rcParams['mathtext.fontset'] = 'stix'

fig = plt.figure(figsize=(10, 6), dpi=dpi_for_pub)
gs = gridspec.GridSpec(1, 3, width_ratios=[0.8, 1, 1])

axes = [fig.add_subplot(gs[0, i]) for i in range(3)]

#updated features
new_feat =  (
    '$_{num}$$\\delta$$^{13C}$',
    '$_{max}$$\\delta$$^{13C}$',
    '$_{mean}$$\\delta$$^{13C}$',
    '$_{skew}$$\\delta$$^{13C}$',
    '$_{kurt}$$\\delta$$^{13C}$',
    '$_{num}$$\\delta$$^{1H}$',
    '$_{max}$$\\delta$$^{1H}$',
    '$_{skew}$$\\delta$$^{1H}$'
)

# Left: Feature importances bar chart
colors = ["#e6308a" if "13C" in f else "#89ce00" if "1H" in f else "#888888" for f in features]
axes[0].barh(new_feat, qm9_model_importances, color=[colors[new_feat.index(f)] for f in new_feat])
axes[0].set_xlabel('Importance', fontsize=13)
axes[0].set_ylabel('Feature', fontsize=13)
axes[0].text(-0.1, 1.0, 'A', transform=axes[0].transAxes, fontsize=16, fontweight='bold', va='top', ha='right')

# SHAP scatter plots
dot_size = 3

shap.plots.scatter(explanation_subset[:, "13C Maximum Shift"], ax=axes[1], 
                   color="#e6308a", dot_size=dot_size, show=False)
axes[1].set_ylabel("SHAP Value", labelpad=-5, loc='center')
axes[1].text(0.1, 1.0, 'B', transform=axes[1].transAxes, fontsize=16, fontweight='bold', va='top', ha='right')
axes[1].set_xlabel(r'Z-score normalized $_{max}$$\delta$$^{13C}$')

shap.plots.scatter(explanation_subset[:, "1H Number of Shifts"], ax=axes[2], 
                   color="#89ce00", dot_size=dot_size, show=False)
axes[2].set_ylabel("SHAP Value", labelpad=-1, loc='center')
axes[2].text(0.1, 1.0, 'C', transform=axes[2].transAxes, fontsize=16, fontweight='bold', va='top', ha='right')
axes[2].set_xlabel(r'Z-score normalized $_{num}$$\delta$$^{1H}$')

plt.tight_layout()
plt.subplots_adjust(wspace=0.2)  # Reduce horizontal space between subplots
plt.savefig(os.path.join(plots_dir, "figure 3.png"), dpi=dpi_for_pub, bbox_inches='tight')
plt.show()

#### Analysis of most important features

In [ ]:
# updated features
top_4_important_features = ['13C Maximum Shift',
                            '13C Mean Shift',
                            '1H Maximum Shift',
                            '1H Number of Shifts'
                           ]

# Compute correlation matrix
importance_corr = x_cv[top_4_important_features].corr(method='pearson')

importance_annot = importance_corr.map(annotate_corr)

# Plot correlation heatmap
plt.figure(figsize=(12, 10), dpi=dpi_for_pub)


# Create a mapping from old variable names to new names
rename_dict = {
    '13C Maximum Shift': '$_{max}$$\\delta$$^{13C}$',
    '13C Mean Shift': '$_{mean}$$\\delta$$^{13C}$',
    '1H Number of Shifts': '$_{num}$$\\delta$$^{1H}$',
    '1H Maximum Shift': '$_{max}$$\\delta$$^{1H}$'
}

importance_corr_renamed = importance_corr.rename(index=rename_dict, columns=rename_dict)
ax = sns.heatmap(importance_corr_renamed, annot=importance_annot, fmt='', cmap='coolwarm', center=0, cbar_kws={'label': 'Pearson Correlation Coefficient'})

plt.tight_layout()
plots_dir = os.path.join(os.getcwd(), "plots")
plt.savefig(os.path.join(plots_dir, "figure s4.png"), dpi=dpi_for_pub, bbox_inches='tight')
plt.show()

In [ ]:
# SHAP Beeswarm plot
# Get indices of selected features
important_shap_indices_subset = [features.index(f) for f in top_4_important_features]

feature_name_map = {
   '13C Maximum Shift': '$_{max}$$\\delta$$^{13C}$',
    '13C Mean Shift': '$_{mean}$$\\delta$$^{13C}$',
    '1H Number of Shifts': '$_{num}$$\\delta$$^{1H}$',
    '1H Maximum Shift': '$_{max}$$\\delta$$^{1H}$'
}

# Slice values and data for those features
important_shap_values_subset = qm9_explanation.values[:, important_shap_indices_subset]
important_shap_data_subset = X_train_scaled[0:N_ROWS, important_shap_indices_subset]

most_important_explanation_subset = shap.Explanation(
    values=important_shap_values_subset,
    base_values=qm9_explanation.base_values,
    data=important_shap_data_subset,
    feature_names=top_4_important_features
)

# Apply names for plotting
most_important_explanation_subset.feature_names = [
    feature_name_map.get(f, f) for f in most_important_explanation_subset.feature_names
]

shap.plots.beeswarm(most_important_explanation_subset, show=False)
plt.xlabel('SHAP value')
plt.tight_layout()
plt.subplots_adjust(wspace=0.2)  # Reduce horizontal space between subplots
plt.savefig(os.path.join(plots_dir, "figure s5.png"), dpi=dpi_for_pub, bbox_inches='tight')
plt.show()

### figure 4

In [ ]:
oxidation_df = qm9_feat.copy()

# find all fragment columns that start with 'fr_'
fr_cols = [c for c in oxidation_df.columns if c.startswith('fr_')]

# fill missing with 0 for safety
oxidation_df[fr_cols] = oxidation_df[fr_cols].fillna(0)

# masks
mask_all_zero = (oxidation_df[fr_cols] == 0).all(axis=1)

mask_only_alcohol = (
    ((oxidation_df.get('fr_Al_OH', 0) == 1) | (oxidation_df.get('fr_Ar_OH', 0) == 1)) &
    (oxidation_df[[c for c in fr_cols if c not in ['fr_Al_OH', 'fr_Ar_OH']]] == 0).all(axis=1)
)

ketone_cols = ['fr_C_O', 'fr_C_O_noCOO', 'fr_ketone', 'fr_ketone_Topliss'] 
not_ketone = [c for c in fr_cols if c not in ketone_cols]
mask_only_ketone = (oxidation_df[ketone_cols] == 1).all(axis=1) & (oxidation_df[not_ketone] == 0).all(axis=1)

aldehyde_cols = ['fr_C_O', 'fr_C_O_noCOO', 'fr_aldehyde']  
not_aldehyde = [c for c in fr_cols if c not in aldehyde_cols]
mask_only_aldehyde = (oxidation_df[aldehyde_cols] == 1).all(axis=1) & (oxidation_df[not_aldehyde] == 0).all(axis=1)

feature_ox = "13C Maximum Shift"

# category column
oxidation_df['fr_category'] = np.select(
    [mask_all_zero, mask_only_alcohol, mask_only_ketone, mask_only_aldehyde], 
    ['No Heteroatoms', 'Alcohol', 'Ketone', 'Aldehyde'],
    default='other_fr'
)

# numeric codes if needed
oxidation_df['fr_category_code'] = oxidation_df['fr_category'].map({'No Heteroatoms': 0,
                                                        'Alcohol': 1,
                                                        'Ketone': 2,
                                                        'Aldehyde': 3
                                                        })

# remove all other fgs
oxidation_df = oxidation_df[oxidation_df['fr_category'] != 'other_fr']

# group means on both axes
oxidation_means = oxidation_df.groupby('fr_category').agg(mean_x=(feature_ox,'mean'),
                                           std_x=(feature_ox,'std'),
                                           mean_y=('gap_ev','mean'),
                                           std_y=('gap_ev','std'),
                                           n=(feature_ox,'size')).reset_index()

In [ ]:
unsaturation_df = qm9_feat.copy()

# calculate degree of unsaturation
unsaturation_df['n_F'] = unsaturation_df['atoms'].apply(lambda s: s.count('F'))
unsaturation_df['n_C'] = unsaturation_df['atoms'].apply(lambda s: s.count('C'))
unsaturation_df['n_H'] = unsaturation_df['atoms'].apply(lambda s: s.count('H'))
unsaturation_df['n_N'] = unsaturation_df['atoms'].apply(lambda s: s.count('N'))

unsaturation_df['unsat_degree'] = (unsaturation_df['n_C'] + 2 + unsaturation_df['n_N'] - unsaturation_df['n_H'] - unsaturation_df['n_F']) / 2

bins = np.arange(-5, 5, 1)  # Bins from -5 to 10, step 1
labels = [f"{i} to {i+1}" for i in bins[:-1]]

unsaturation_df['unsat_degree_cat'] = pd.cut(
    unsaturation_df['unsat_degree'],
    bins=bins,
    labels=labels,
    include_lowest=True
)

unsat_var = 'unsat_degree_cat'
unsat_feature = '1H Number of Shifts'

# compute group means
unsaturation_group_means = unsaturation_df.groupby(unsat_var).agg(
    mean_x=(unsat_feature,'mean'),
    std_x=(unsat_feature,'std'),
    mean_y=('gap_ev','mean'),
    std_y=('gap_ev','std'),
    count=(unsat_feature,'size')
).reset_index()

In [ ]:
fig = plt.figure(figsize=(12, 10), dpi=dpi_for_pub)
plt.rcParams['font.size'] = 11
gs = gridspec.GridSpec(
    3, 2,  # 3 rows, 2 columns
    width_ratios=[2, 1],  # Left column is 3x wider than right
    wspace=0.2, hspace=0.25
)

# Create a custom legend handle for the group average
group_avg_handle = mlines.Line2D(
    [], [], 
    color='black', 
    marker='D', 
    linestyle='None', 
    markersize=3, 
    markerfacecolor='white', 
    markeredgewidth=0.5, 
    label='Group Average'
)

# oxidation
ax0 = fig.add_subplot(gs[0, 0])  
oxidation_df_sorted = oxidation_df.sort_values('fr_category_code')
oxidation_palette = ['#888888', '#d69677', '#a46bce', '#d6bc41']
sns.scatterplot(
    data=oxidation_df_sorted, x=feature_ox, y='gap_ev',
    hue='fr_category',
    palette=oxidation_palette,
    alpha=0.7, s=20, edgecolor='none', ax=ax0
)

sorted_categories = oxidation_df_sorted['fr_category'].unique()
cat_to_color = dict(zip(sorted_categories, oxidation_palette))

for _, r in oxidation_means.iterrows():
    # Mean point
    ax0.scatter(
        r['mean_x'], r['mean_y'],
        s=100,
        facecolors=cat_to_color[r['fr_category']],
        edgecolors='black',
        linewidths=1.2,
        zorder=10,
        marker="D"
    )
    # Horizontal bar for std in y
    ax0.plot([r['mean_x'], r['mean_x']], [r['mean_y']-r['std_y'], r['mean_y']+r['std_y']],
             color='black', linestyle='-', linewidth=1, alpha=1, zorder=9)
    # Add horizontal caps at the ends
    cap_width = 0.8  # adjust as needed
    ax0.plot([r['mean_x']-cap_width, r['mean_x']+cap_width], [r['mean_y']+r['std_y'], r['mean_y']+r['std_y']], color='black', linewidth=1)
    ax0.plot([r['mean_x']-cap_width, r['mean_x']+cap_width], [r['mean_y']-r['std_y'], r['mean_y']-r['std_y']], color='black', linewidth=1)

    # Vertical bar for std in x
    ax0.plot([r['mean_x']-r['std_x'], r['mean_x']+r['std_x']], [r['mean_y'], r['mean_y']],
             color='black', linestyle='-', linewidth=1, alpha=1, zorder=9)
    cap_height = 0.1  # adjust as needed
    ax0.plot([r['mean_x']-r['std_x'], r['mean_x']-r['std_x']], [r['mean_y']-cap_height, r['mean_y']+cap_height], color='black', linewidth=1)
    ax0.plot([r['mean_x']+r['std_x'], r['mean_x']+r['std_x']], [r['mean_y']-cap_height, r['mean_y']+cap_height], color='black', linewidth=1)

# tidy legend: keep only hue entries
handles, labels = ax0.get_legend_handles_labels()
n_hue = len(oxidation_means)

# Add the custom handle to the legend
handles.append(group_avg_handle)
labels.append('Group Average')

ax0.set_xlabel(r'$_{max}$$\delta$$^{13C}$')
ax0.set_ylabel('B3LYP/6-31G(2df,p) computed $\\Delta$E (eV)')
ymin, ymax = ax0.get_ylim()
ax0.set_yticks(np.arange(np.floor(ymin), np.ceil(ymax) + 2, 2))
ax0.legend(
    handles=handles,
    labels=labels,
    title='Functional Group',
    bbox_to_anchor=(1.00, 1.00),
    loc='upper right',
    fontsize=8,                # Smaller text
    title_fontsize=9,          # Smaller title
    borderaxespad=0.2,         # Compress legend closer to axes
    markerscale=2.0           # Smaller legend markers
)

# unsaturation
unsaturation_df_sorted = unsaturation_df.sort_values('unsat_degree_cat')
unsaturation_palette = ['#c0e23f', '#64d973', '#30d389',
'#b9eecf', '#98e5ba', '#32d9d2',
'#46b7fa', '#8573ff', '#5047cb']
hue_order = unsaturation_df_sorted[unsat_var].unique()
unsaturation_cat_to_color = dict(zip(hue_order, unsaturation_palette))

ax2 = fig.add_subplot(gs[1, 0])  
ax2 = sns.scatterplot(data=unsaturation_df_sorted,
                     x=unsat_feature,
                     y='gap_ev',
                     hue=unsat_var,
                     palette=unsaturation_palette,
                     alpha=0.8,
                     s=20,
                     edgecolor='none')

# annotate means with class and count
for _, r in unsaturation_group_means.iterrows():
    ax2.scatter(
        r['mean_x'], r['mean_y'],
        s=100,
        facecolors=unsaturation_cat_to_color[r[unsat_var]],
        edgecolors='black',
        linewidths=1.2,
        marker='D',
        zorder=10
    )
    # Horizontal bar for std in y
    ax2.plot([r['mean_x'], r['mean_x']], [r['mean_y']-r['std_y'], r['mean_y']+r['std_y']],
             color='black', linestyle='-', linewidth=1, alpha=1, zorder=9)
    # Add horizontal caps at the ends
    cap_width = 0.08  # adjust as needed
    ax2.plot([r['mean_x']-cap_width, r['mean_x']+cap_width], [r['mean_y']+r['std_y'], r['mean_y']+r['std_y']], color='black', linewidth=1)
    ax2.plot([r['mean_x']-cap_width, r['mean_x']+cap_width], [r['mean_y']-r['std_y'], r['mean_y']-r['std_y']], color='black', linewidth=1)

    # Vertical bar for std in x
    ax2.plot([r['mean_x']-r['std_x'], r['mean_x']+r['std_x']], [r['mean_y'], r['mean_y']],
             color='black', linestyle='-', linewidth=1, alpha=1, zorder=9)
    cap_height = 0.17  # adjust as needed
    ax2.plot([r['mean_x']-r['std_x'], r['mean_x']-r['std_x']], [r['mean_y']-cap_height, r['mean_y']+cap_height], color='black', linewidth=1)
    ax2.plot([r['mean_x']+r['std_x'], r['mean_x']+r['std_x']], [r['mean_y']-cap_height, r['mean_y']+cap_height], color='black', linewidth=1)


ax2.set_xlabel(r'$_{num}$$\delta$$^{1H}$')
ax2.set_ylabel('B3LYP/6-31G(2df,p) computed $\\Delta$E (eV)')
ymin, ymax = ax2.get_ylim()
ax2.set_yticks(np.arange(np.floor(ymin), np.ceil(ymax) + 2, 2))

# Get existing handles and labels
handles, labels = ax2.get_legend_handles_labels()
n_hue = len(handles)  # or len(unsaturation_stats) if you want only hue entries

# Add the custom handle to the legend
handles.append(group_avg_handle)
labels.append('Group Average')

ax2.legend(
    handles=handles,
    labels=labels,
    title='Degree of Unsaturation',
    bbox_to_anchor=(1.00, 0.0),
    loc='lower right',
    fontsize=8,
    title_fontsize=9,
    borderaxespad=0.2,
    markerscale=2.0,
    ncol=2
)

plots_dir = os.path.join(os.getcwd(), "plots")
fig.savefig(os.path.join(plots_dir, "figure 4.png"), dpi=dpi_for_pub, bbox_inches='tight')
plt.show()

In [ ]:
# For each functional group, get the molecule(s) with the highest 13C mean shift
N = 1  # Number of top molecules per group (adjust as needed)

fg_masks = {'alcohol': mask_only_alcohol, 'aldehyde': mask_only_aldehyde, 'ketone': mask_only_ketone}
# Make sure fg_masks is defined as a dict mapping group names to boolean masks
highest_C13_by_fg = {}
for fg_name, mask in fg_masks.items():
    subset = oxidation_df[mask]
    if not subset.empty:
        top = subset.sort_values('13C Maximum Shift', ascending=True).head(N)
        highest_C13_by_fg[fg_name] = top

# Combine results into a single DataFrame for plotting
plot_df = pd.concat(highest_C13_by_fg.values(), keys=highest_C13_by_fg.keys())

# Plot molecules for each functional group
for fg_name, group_df in highest_C13_by_fg.items():
    mols = [Chem.MolFromSmiles(smiles) for smiles in group_df['smiles']]
    legends = [f"{fg_name}\nC13: {c13:.2f}\nGap: {gap:.2f}" for c13, gap in zip(group_df['13C Maximum Shift'], group_df['gap_ev'])]
    img = Draw.MolsToGridImage(mols, legends=legends, molsPerRow=1, subImgSize=(250,250), returnPNG=False)
    plt.figure(figsize=(3,3))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f"Highest 13C shift: {fg_name}")
    plt.show()

In [ ]:
# Get the 3 most and 3 least unsaturated molecules
most_unsat = unsaturation_df.nlargest(3, 'unsat_degree')
least_unsat = unsaturation_df.nsmallest(3, 'unsat_degree')

# Convert SMILES to RDKit Mol objects
most_mols = [Chem.MolFromSmiles(sm) for sm in most_unsat['smiles']]
least_mols = [Chem.MolFromSmiles(sm) for sm in least_unsat['smiles']]

most_legends = [
    f"Unsat: {row['unsat_degree']:.2f}\nGap: {row['gap_ev']:.2f}\n1H shifts: {row['1H Number of Shifts']}"
    for _, row in most_unsat.iterrows()
]
least_legends = [
    f"Unsat: {row['unsat_degree']:.2f}\nGap: {row['gap_ev']:.2f}\n1H shifts: {row['1H Number of Shifts']}"
    for _, row in least_unsat.iterrows()
]

# Plot using RDKit
img_most = Draw.MolsToGridImage(most_mols, molsPerRow=3, subImgSize=(200,200), legends=most_legends, returnPNG=False)
img_least = Draw.MolsToGridImage(least_mols, molsPerRow=3, subImgSize=(200,200), legends=least_legends, returnPNG=False)

In [ ]:
img_most

In [ ]:
img_least

### figure s3

In [ ]:
# all shap plots
mpl.rcParams.update({'font.size': 10})
mpl.rcParams['mathtext.fontset'] = 'stix'

fig = plt.figure(figsize=(12, 10), dpi=dpi_for_pub)
gs = gridspec.GridSpec(3, 3, wspace=0.3, hspace=0.3)  # 3 rows, 3 columns

axes = [fig.add_subplot(gs[i, j]) for i in range(3) for j in range(3)]

#updated features
new_feat =  (
    '$_{num}$$\\delta$$^{13C}$',
    '$_{max}$$\\delta$$^{13C}$',
    '$_{mean}$$\\delta$$^{13C}$',
    '$_{skew}$$\\delta$$^{13C}$',
    '$_{kurt}$$\\delta$$^{13C}$',
    '$_{num}$$\\delta$$^{1H}$',
    '$_{max}$$\\delta$$^{1H}$',
    '$_{skew}$$\\delta$$^{1H}$'
)

# Left: Feature importances bar chart
colors = ["#e6308a" if "13C" in f else "#89ce00" if "1H" in f else "#888888" for f in new_feat]
axes[0].barh(new_feat, qm9_model_importances, color=[colors[new_feat.index(f)] for f in new_feat])
axes[0].set_xlabel('Importance', fontsize=13)
axes[0].set_ylabel('Feature', fontsize=13)
axes[0].text(-0.15, 1.0, 'A', transform=axes[0].transAxes, fontsize=16, fontweight='bold', va='top', ha='right')

# SHAP scatter plots
dot_size = 3

shap.plots.scatter(full_explanation[0:N_ROWS, "1H Skewness"], ax=axes[1], 
                   color="#89ce00", dot_size=dot_size, show=False)
axes[1].set_ylabel("SHAP Value", labelpad=-1, loc='center')
axes[1].text(0.1, 1.0, 'B', transform=axes[1].transAxes, fontsize=16, fontweight='bold', va='top', ha='right')
axes[1].set_xlabel(r'Z-score normalized $_{skew}$$\delta$$^{1H}$')

shap.plots.scatter(full_explanation[0:N_ROWS, "1H Maximum Shift"], ax=axes[2], 
                   color="#89ce00", dot_size=dot_size, show=False)
axes[2].set_ylabel("SHAP Value", labelpad=-1, loc='center')
axes[2].text(0.1, 1.0, 'C', transform=axes[2].transAxes, fontsize=16, fontweight='bold', va='top', ha='right')
axes[2].set_xlabel(r'Z-score normalized $_{max}$$\delta$$^{1H}$')

shap.plots.scatter(full_explanation[0:N_ROWS, "1H Number of Shifts"], ax=axes[3], 
                   color="#89ce00", dot_size=dot_size, show=False)
axes[3].set_ylabel("SHAP Value", labelpad=-1, loc='center')
axes[3].text(0.1, 1.0, 'D', transform=axes[3].transAxes, fontsize=16, fontweight='bold', va='top', ha='right')
axes[3].set_xlabel(r'Z-score normalized $_{num}$$\delta$$^{1H}$')

shap.plots.scatter(full_explanation[0:N_ROWS, "13C Kurtosis"], ax=axes[4], 
                   color="#e6308a", dot_size=dot_size, show=False)
axes[4].set_ylabel("SHAP Value", labelpad=-1, loc='center')
axes[4].text(0.1, 1.0, 'E', transform=axes[4].transAxes, fontsize=16, fontweight='bold', va='top', ha='right')
axes[4].set_xlabel(r'Z-score normalized $_{kurt}$$\delta$$^{13C}$')

shap.plots.scatter(full_explanation[0:N_ROWS, "13C Skewness"], ax=axes[5], 
                   color="#e6308a", dot_size=dot_size, show=False)
axes[5].set_ylabel("SHAP Value", labelpad=-1, loc='center')
axes[5].text(0.1, 1.0, 'F', transform=axes[5].transAxes, fontsize=16, fontweight='bold', va='top', ha='right')
axes[5].set_xlabel(r'Z-score normalized $_{skew}$$\delta$$^{13C}$')

shap.plots.scatter(full_explanation[0:N_ROWS, "13C Mean Shift"], ax=axes[6], 
                   color="#e6308a", dot_size=dot_size, show=False)
axes[6].set_ylabel("SHAP Value", labelpad=-1, loc='center')
axes[6].text(0.1, 1.0, 'G', transform=axes[6].transAxes, fontsize=16, fontweight='bold', va='top', ha='right')
axes[6].set_xlabel(r'Z-score normalized $_{mean}$$\delta$$^{13C}$')

shap.plots.scatter(full_explanation[0:N_ROWS, "13C Maximum Shift"], ax=axes[7], 
                   color="#e6308a", dot_size=dot_size, show=False)
axes[7].set_ylabel("SHAP Value", labelpad=-1, loc='center')
axes[7].text(0.1, 1.0, 'H', transform=axes[7].transAxes, fontsize=16, fontweight='bold', va='top', ha='right')
axes[7].set_xlabel(r'Z-score normalized $_{max}$$\delta$$^{13C}$')

shap.plots.scatter(full_explanation[0:N_ROWS, "13C Number of Shifts"], ax=axes[8], 
                   color="#e6308a", dot_size=dot_size, show=False)
axes[8].set_ylabel("SHAP Value", labelpad=-1, loc='center')
axes[8].text(0.1, 1.0, 'I', transform=axes[8].transAxes, fontsize=16, fontweight='bold', va='top', ha='right')
axes[8].set_xlabel(r'Z-score normalized $_{num}$$\delta$$^{13C}$')

plt.subplots_adjust(wspace=0.2)  # Reduce horizontal space between subplots
plt.savefig(os.path.join(plots_dir, "figure s3.png"), dpi=dpi_for_pub, bbox_inches='tight')
plt.show()

# NMRShiftDB2

## NMRShiftDB2 Data Extraction

In [ ]:
# load nmrshiftdb2
sdf_file = # enter appropriate directory
nmrshiftdb2 = funs.load_sdf_to_df(sdf_file)
print(f"Number of entries in NMRShiftDB2 dataset: {len(nmrshiftdb2)}")
print(f"Shape of NMRShiftDB2 dataframe: {nmrshiftdb2.shape}")

## NMRShiftDB2 Data Featurization

In [ ]:
# removal of unused columns
nmrshiftdb2_feat = nmrshiftdb2.loc[:, ~nmrshiftdb2.columns.str.contains("NMRBasisSet|Assignment Method|NMRLocalis|NMRStandard|GeomMethod|GeomBasisSet|NMRMethod|Machine|Program|19F|J-|195Pt|33S|31P|17O|ROESY|HMQC|DEPTQ|HSQC|HMBC|voltammetry|rawdata|COSY|NOESY|TOCSY|APT|11B|DEPT|73Ge|29Si|15N")]
print(f'A total of {nmrshiftdb2.shape[1]-nmrshiftdb2_feat.shape[1]} columns were removed')

In [ ]:
# add smiles by converting from rdkit_mol
nmrshiftdb2_feat = nmrshiftdb2_feat.copy()
nmrshiftdb2_feat['smiles'] = nmrshiftdb2_feat['rdkit_mol'].apply(lambda mol: Chem.MolToSmiles(mol) if mol is not None else None)

In [ ]:
# are there cases where Spectrum 13C 0 is NaN but there are other Spectrum 13C columns that are not NaN?
# if no, this allows us to only take rows where Spectrum 13C 0 is not NaN
cols = [f"Spectrum 13C {i}" for i in range(1, 11)]
mask = (
    nmrshiftdb2_feat["Spectrum 13C 0"].isna() &
    nmrshiftdb2_feat[cols].notna().any(axis=1)
)
print(f"Number of such cases: {mask.sum()}")

In [ ]:
# Remove entries that do not have any 13C NMR
nmrshiftdb2_feat = nmrshiftdb2_feat[nmrshiftdb2_feat['Spectrum 13C 0'].notnull()]
print(f'There are {nmrshiftdb2_feat.shape[0]} entries with carbon NMR spectra')

In [ ]:
# add constitutional information to implement filters
nmrshiftdb2_feat[['molecular_formula',
                  'mol_weight',
                  'num_heavy_atoms',
                  'num_carbons',
                  'num_hydrogens']] = nmrshiftdb2_feat['rdkit_mol'].apply(funs.get_rdkit_constitutional)

# Filter for molecules with at least 4 H and 4 C
nmrshiftdb2_feat = nmrshiftdb2_feat[(nmrshiftdb2_feat["num_hydrogens"] >= 4) & (nmrshiftdb2_feat["num_carbons"] >= 4)]
print(f"A total of {len(nmrshiftdb2_feat)} molecules have at least 4 C and 4 H atoms.")

# only take molecules with molecular weight below 500 g/mol
nmrshiftdb2_feat = nmrshiftdb2_feat[nmrshiftdb2_feat["mol_weight"] < 500.0]
print(f"A total of {len(nmrshiftdb2_feat)} molecules have molecular weight below 500 g/mol.")

In [ ]:
# specify columns for 13C and 1H
C13_cols = [col for col in nmrshiftdb2_feat.columns if col.startswith('Spectrum 13C')]
H1_cols = [col for col in nmrshiftdb2_feat.columns if col.startswith('Spectrum 1H')]

# Columns to inherit
meta_cols = [
    'INChI key', 'INChI', 'nmrshiftdb2 ID', 'rdkit_mol', 'smiles', 'cas', 'synonyms',
    'additional_information', 'molecular_formula', 'mol_weight', 'num_heavy_atoms',
    'num_carbons', 'num_hydrogens', 'Temperature [K]', 'Field Strength [MHz]', 'Solvent'
]

In [ ]:
# get reformatted rows for 13C and 1H separately
# 13C
rows = []
for _, row in nmrshiftdb2_feat.iterrows():
    for col in C13_cols:
        value = row[col]
        if pd.notna(value):
            idx = funs.get_index(col)
            # Build new row
            # first append information from retained columns
            new_row = {meta: row[meta] for meta in meta_cols if meta in row}
            # then add spectrum-specific information
            new_row['spectrum_index'] = idx
            new_row['spectrum_13C'] = value
            rows.append(new_row)
            
nmrshiftdb2_feat_13C = pd.DataFrame(rows)

# 1H
rows = []
for _, row in nmrshiftdb2_feat.iterrows():
    for col in H1_cols:
        value = row[col]
        if pd.notna(value):
            idx = funs.get_index(col)
            # Build new row
            # first append information from retained columns
            new_row = {meta: row[meta] for meta in meta_cols if meta in row}
            # then add spectrum-specific information
            new_row['spectrum_index'] = idx
            new_row['spectrum_1H'] = value
            rows.append(new_row)
            
nmrshiftdb2_feat_1H = pd.DataFrame(rows)

In [ ]:
# get index and value pairs for Temperature, Field Strength, and Solvent

# 13C
for col in ['Temperature [K]', 'Field Strength [MHz]', 'Solvent']:
    index_col = f"{col}_index_check"
    value_col = f"{col}_value"
    nmrshiftdb2_feat_13C[[index_col, value_col]] = nmrshiftdb2_feat_13C[col].apply(
        lambda cell: pd.Series(funs.extract_index_value_pairs(cell)
    )  )

# 1H
for col in ['Temperature [K]', 'Field Strength [MHz]', 'Solvent']:
    index_col = f"{col}_index_check"
    value_col = f"{col}_value"
    nmrshiftdb2_feat_1H[[index_col, value_col]] = nmrshiftdb2_feat_1H[col].apply(
        lambda cell: pd.Series(funs.extract_index_value_pairs(cell))
    )

In [ ]:
# drop rows that are not experimental
# this based on the fact that these conditions are only reported for experimental

# 13C
for col in ['Temperature [K]', 'Field Strength [MHz]', 'Solvent']:
    index_col = f"{col}_index_check"
    nmrshiftdb2_feat_13C = nmrshiftdb2_feat_13C[
        nmrshiftdb2_feat_13C.apply(lambda row: int(row['spectrum_index']) in row[index_col], axis=1)
    ]
print(f'There are {nmrshiftdb2_feat_13C.shape[0]} entries in the 13C NMR dataset after filtering for experimental conditions.')

# 1H
for col in ['Temperature [K]', 'Field Strength [MHz]', 'Solvent']:
    index_col = f"{col}_index_check"
    nmrshiftdb2_feat_1H = nmrshiftdb2_feat_1H[
        nmrshiftdb2_feat_1H.apply(lambda row: int(row['spectrum_index']) in row[index_col], axis=1)
    ]
print(f'There are {nmrshiftdb2_feat_1H.shape[0]} entries in the 1H NMR dataset after filtering for experimental conditions.')

In [ ]:
# now overwrite conditions with the corresponding actual condition for that spectrum

# 13C
for col in ['Temperature [K]', 'Field Strength [MHz]', 'Solvent']:
    index_col = f"{col}_index_check"
    value_col = f"{col}_value"
    def get_value_for_index(row):
        idx_list = row[index_col]
        val_list = row[value_col]
        try:
            pos = idx_list.index(int(row['spectrum_index']))
            return val_list[pos]
        except (ValueError, IndexError, TypeError):
            return None
    nmrshiftdb2_feat_13C[col] = nmrshiftdb2_feat_13C.apply(get_value_for_index, axis=1)
print(f'Number of rows (QC check) in 13C NMR dataset after condition assignment: {len(nmrshiftdb2_feat_13C)}')

# 1H
for col in ['Temperature [K]', 'Field Strength [MHz]', 'Solvent']:
    index_col = f"{col}_index_check"
    value_col = f"{col}_value"
    def get_value_for_index(row):
        idx_list = row[index_col]
        val_list = row[value_col]
        try:
            pos = idx_list.index(int(row['spectrum_index']))
            return val_list[pos]
        except (ValueError, IndexError, TypeError):
            return None
    nmrshiftdb2_feat_1H[col] = nmrshiftdb2_feat_1H.apply(get_value_for_index, axis=1)
print(f'Number of rows (QC check) in 1H NMR dataset after condition assignment: {len(nmrshiftdb2_feat_1H)}')

In [ ]:
# drop unneeded columns
nmrshiftdb2_feat_13C = nmrshiftdb2_feat_13C.drop(columns=['Temperature [K]_index_check', 'Field Strength [MHz]_index_check',
                          'Solvent_index_check','Temperature [K]_value','Field Strength [MHz]_value','Solvent_value',
                          'spectrum_index'])
nmrshiftdb2_feat_1H = nmrshiftdb2_feat_1H.drop(columns=['Temperature [K]_index_check', 'Field Strength [MHz]_index_check',
                          'Solvent_index_check','Temperature [K]_value','Field Strength [MHz]_value','Solvent_value',
                          'spectrum_index'])

In [ ]:
# convert spectra to array
nmrshiftdb2_feat_13C = funs.extract_first_floats(nmrshiftdb2_feat_13C, 'spectrum_13C')
nmrshiftdb2_feat_1H = funs.extract_first_floats(nmrshiftdb2_feat_1H, 'spectrum_1H')

print(f'Number of rows (QC check) in 13C NMR dataset after array conversion: {len(nmrshiftdb2_feat_13C)}')
print(f'Number of rows (QC check) in 1H NMR dataset after array conversion: {len(nmrshiftdb2_feat_1H)}')

In [ ]:
# qc step to remove rows where number of carbons does not match number of signals
# we're doing this because the prediction method requires each carbon with its own value
# note that this may remove molecules where there are equivalent carbons that have integration values greater than 1
# no matching conducted for 1H NMR, this is too challenging due to OH, NH, etc

signal_col = 'spectrum_13C'
count_col = 'num_carbons'
full_13C = nmrshiftdb2_feat_13C.copy()
nmrshiftdb2_feat_13C = nmrshiftdb2_feat_13C[
    nmrshiftdb2_feat_13C.apply(lambda row: funs.signals_match(row, signal_col, count_col), axis=1)
]
nmrshiftdb2_feat_13C_unmatched = full_13C.loc[full_13C.index.difference(nmrshiftdb2_feat_13C.index)]
print(f'There are {len(nmrshiftdb2_feat_13C_unmatched)} entries where the number of 13C NMR signals does not match the number of carbons in the molecule.')

print(f'Number of rows (QC check) in 13C NMR dataset after signal number matching: {len(nmrshiftdb2_feat_13C)}')

# now see how many duplicate spectra there are
dup_mask_13C = nmrshiftdb2_feat_13C.duplicated(subset=['INChI key'], keep=False)
num_dup_rows_13C = dup_mask_13C.sum()

dup_mask_1H = nmrshiftdb2_feat_1H.duplicated(subset=['INChI key'], keep=False)
num_dup_rows_1H = dup_mask_1H.sum()

print(f"Number of 13C NMR rows with duplicate 'INChI key': {num_dup_rows_13C}")
print(f"Number of 1H NMR rows with duplicate 'INChI key': {num_dup_rows_1H}")

In [ ]:
# Where there are duplicate spectra, prefer to take only the spectra with solvent "Chloroform-D1 (CDCl3)"
# prefer rows with "CDCl3" secondarily
# If there are no rows with this solvent, or there are more than one row with this solvent,
# take only the first row for each duplicate.
nmrshiftdb2_feat_13C_deduped = funs.deduplicate_spectra(nmrshiftdb2_feat_13C, id_col='INChI key', 
                                                        preferred_solvent="Chloroform-D1 (CDCl3)",
                                                        second_preferred_solvent='CDCl3')
nmrshiftdb2_feat_1H_deduped = funs.deduplicate_spectra(nmrshiftdb2_feat_1H, id_col='INChI key', 
                                                       preferred_solvent="Chloroform-D1 (CDCl3)",
                                                        second_preferred_solvent='CDCl3')
print(f"After deduplication, there are {len(nmrshiftdb2_feat_13C_deduped)} unique 13C NMR entries.")
print(f"After deduplication, there are {len(nmrshiftdb2_feat_1H_deduped)} unique 1H NMR entries.")

# verify duplicates dropped
dup_mask_13C = nmrshiftdb2_feat_13C_deduped.duplicated(subset=['INChI key'], keep=False)
num_dup_rows_13C = dup_mask_13C.sum()
print(f"Number of 13C NMR rows with duplicate 'INChI key': {num_dup_rows_13C}")

dup_mask_1H = nmrshiftdb2_feat_1H_deduped.duplicated(subset=['INChI key'], keep=False)
num_dup_rows_1H = dup_mask_1H.sum()
print(f"Number of 1H NMR rows with duplicate 'INChI key': {num_dup_rows_1H}")

In [ ]:
# Filter out invalid molecules
nmrshiftdb2_feat_13C_deduped = nmrshiftdb2_feat_13C_deduped[nmrshiftdb2_feat_13C_deduped['rdkit_mol'].apply(funs.is_valid_mol)]
print(f"After removing invalid valence: {nmrshiftdb2_feat_13C_deduped.shape[0]} entries remain in 13C NMR dataset.")

nmrshiftdb2_feat_1H_deduped = nmrshiftdb2_feat_1H_deduped[nmrshiftdb2_feat_1H_deduped['rdkit_mol'].apply(funs.is_valid_mol)]
print(f"After removing invalid valence: {nmrshiftdb2_feat_1H_deduped.shape[0]} entries remain in 1H NMR dataset.")

In [ ]:
# remove heavy atoms
atoms_to_remove = ["Pb", "Se", "Zn", "Si", "Sn", "Fe"]
pattern = "|".join(atoms_to_remove)

nmrshiftdb2_feat_13C_clean = nmrshiftdb2_feat_13C_deduped[~nmrshiftdb2_feat_13C_deduped['molecular_formula'].str.contains(pattern, na=False)]
print(f"After removing heavy atoms: {nmrshiftdb2_feat_13C_clean.shape[0]} entries remain in 13C NMR dataset.")

nmrshiftdb2_feat_1H_clean = nmrshiftdb2_feat_1H_deduped[~nmrshiftdb2_feat_1H_deduped['molecular_formula'].str.contains(pattern, na=False)]
print(f"After removing heavy atoms: {nmrshiftdb2_feat_1H_clean.shape[0]} entries remain in 1H NMR dataset.")

In [ ]:
# drop unneeded columns again
nmrshiftdb2_feat_13C_clean = nmrshiftdb2_feat_13C_clean.drop(columns=['solvent_priority'])
nmrshiftdb2_feat_1H_clean = nmrshiftdb2_feat_1H_clean.drop(columns=['solvent_priority'])

In [ ]:
# Remove rows where the variance of the spectrum_13C array is < 0.001
# this causes a failure of kurtosis and skewness values
def array_variance(arr):
    arr = np.array(arr)
    if arr.size == 0:
        return np.nan
    return np.var(arr)
mask_13C = nmrshiftdb2_feat_13C_clean['spectrum_13C'].apply(array_variance) >= 0.001
mask_1H = nmrshiftdb2_feat_1H_clean['spectrum_1H'].apply(array_variance) >= 0.001
nmrshiftdb2_feat_13C_stats = nmrshiftdb2_feat_13C_clean[mask_13C].reset_index(drop=True)
nmrshiftdb2_feat_1H_stats = nmrshiftdb2_feat_1H_clean[mask_1H].reset_index(drop=True)
print(f'Number of rows remaining after variance filter: {len(nmrshiftdb2_feat_13C_stats)}')
print(f'Number of rows remaining after variance filter: {len(nmrshiftdb2_feat_1H_stats)}')

In [ ]:
# compute features
nmrshiftdb2_feat_13C_stats = funs.get_summary_stats(nmrshiftdb2_feat_13C_stats, array_col="spectrum_13C")
nmrshiftdb2_feat_1H_stats = funs.get_summary_stats(nmrshiftdb2_feat_1H_stats, array_col="spectrum_1H")

In [ ]:
# get combined dataset
nmrshift_13C_1H = nmrshiftdb2_feat_13C_stats.merge(nmrshiftdb2_feat_1H_stats[['INChI key', 
                                                                             'spectrum_1H_length',
                                                                            'spectrum_1H_min',
                                                                            'spectrum_1H_max',
                                                                            'spectrum_1H_mean',
                                                                            'spectrum_1H_median',
                                                                            'spectrum_1H_mode',
                                                                            'spectrum_1H_var',
                                                                            'spectrum_1H_std',
                                                                            'spectrum_1H_skewness',
                                                                            'spectrum_1H_kurtosis']], 
                                                    on='INChI key', how='left').dropna(subset=['spectrum_1H_length'])

print(f'There are {nmrshift_13C_1H.shape[0]} molecules with both 13C and 1H NMR shifts.')

In [ ]:
# add rdkit functional groups
nmrshift_13C_1H = funs.get_all_fgs(nmrshift_13C_1H, 'smiles')
nmrshift_13C_1H = funs.get_lipinski_descriptors(nmrshift_13C_1H, 'smiles')

In [ ]:
# get output properties from Gaussian calculations
nmrshift_13C_1H_gaussian = funs.process_directory() # enter appropriate directory

In [ ]:
# merge all features and properties
nmrshift_13C_1H_feat = nmrshift_13C_1H.merge(nmrshift_13C_1H_gaussian, 
                                                    left_on='INChI key', right_on='name', how='left').dropna(subset=['gap_ev']).dropna(subset=['spectrum_13C_length'])
print(f'There are {nmrshift_13C_1H_feat.shape[0]} molecules with 13C spectra and a calculated HOMO-LUMO gap')

In [ ]:
# screen by atoms
atoms_to_remove = ["As", "Ge"]
pattern = "|".join(atoms_to_remove)
nmrshift_13C_1H_feat = nmrshift_13C_1H_feat[~nmrshift_13C_1H_feat['molecular_formula'].str.contains(pattern, na=False)]
print(f"After removing As, Ge: {nmrshift_13C_1H_feat.shape[0]} entries remain.")

# get current atoms in all molecular formulas
formulas = nmrshift_13C_1H_feat['molecular_formula'].dropna().astype(str)
# Regex to find atom symbols (capital letter followed by optional lowercase letter)
atom_pattern = r'([A-Z][a-z]?)'
atoms = set()
for formula in formulas:
    atoms.update(re.findall(atom_pattern, formula))
print(sorted(atoms))

In [ ]:
# Dataset saving checkpoint
os.chdir("data")
nmrshift_13C_1H_feat.to_pickle("nmrshift_13C_1H_feat.pkl")
os.chdir("..")

## NMRShiftDB2 Feature Selection

In [ ]:
# rename variables
nmrshift_13C_1H_feat = nmrshift_13C_1H_feat.rename(columns={
    'spectrum_13C_length': '13C Number of Shifts',
    'spectrum_13C_min': '13C Minimum Shift',
    'spectrum_13C_max': '13C Maximum Shift',
    'spectrum_13C_mean': '13C Mean Shift',
    'spectrum_13C_median': '13C Median Shift',
    'spectrum_13C_mode': '13C Mode Shift',
    'spectrum_13C_var': '13C Variance',
    'spectrum_13C_std': '13C Standard Deviation',
    'spectrum_13C_skewness': '13C Skewness',
    'spectrum_13C_kurtosis': '13C Kurtosis',
    'spectrum_1H_length': '1H Number of Shifts',
    'spectrum_1H_min': '1H Minimum Shift',
    'spectrum_1H_max': '1H Maximum Shift',
    'spectrum_1H_mean': '1H Mean Shift',
    'spectrum_1H_median': '1H Median Shift',
    'spectrum_1H_mode': '1H Mode Shift',
    'spectrum_1H_var': '1H Variance',
    'spectrum_1H_std': '1H Standard Deviation',
    'spectrum_1H_skewness': '1H Skewness',
    'spectrum_1H_kurtosis': '1H Kurtosis'
})

In [ ]:
# specify features, taken from those used for qm9 prediction
# this could potentially be improved upon by doing another round of feature selection
features = ['13C Number of Shifts', '13C Maximum Shift', '13C Mean Shift', '13C Skewness', '13C Kurtosis',
           '1H Number of Shifts', '1H Maximum Shift', '1H Skewness',
           ]

In [ ]:
# get x and y for stratified splitting
X_exp = nmrshift_13C_1H_feat[features]
Y_exp = nmrshift_13C_1H_feat['gap_ev']

## NMRShiftDB2 CV/Test Split

In [ ]:
# create discretized bins for stratification
number_of_bins = 100
Y_binned_exp = pd.qcut(Y_exp, q=number_of_bins, labels=False, duplicates='drop')

# Stratified split based on HOMO-LUMO gap
# select 20% for test set (without replacement)
x_train_exp, x_test_exp, y_train_exp, y_test_exp = train_test_split(X_exp, Y_exp, test_size=0.2, random_state=random_seed, stratify=Y_binned_exp)

print(f'Train set size: {len(x_train_exp)}')
print(f'Test set size: {len(x_test_exp)}')
print(f'Train set size to feature: {len(x_train_exp)/len(features)}')

In [ ]:
# make sure these molecules are in the test set
selected_molecules = [
    ('O=C1c2ccccc2C(=O)c2c1ccc(O)c2O'), #"Alizarin - red dye", 
    ('O=C(O)/C=C/c1ccc(O)cc1'), # "p-Coumaric acid - UV-active"
    ('CC(C)=C/C=C/C(C)=C/C=C/C(C)=C1\C(=O)C[C@H]2[C@@]3(C)CC[C@@H](O)[C@](C)(CO)[C@@H]3CC[C@]12C'), # Stellettin J
]

In [ ]:
# build a lookup so we can find these molecules
smiles_by_idx = nmrshift_13C_1H_feat["smiles"]

# Find selected molecules currently in train
train_smiles = smiles_by_idx.loc[x_train_exp.index]
train_selected_idx = train_smiles[train_smiles.isin(selected_molecules)].index.tolist()

rng = np.random.default_rng(random_seed)

# now make swaps
swaps_done = 0

for idx_train_sel in train_selected_idx:
    # Recompute test candidates each loop (exclude selected molecules from being swapped out)
    test_smiles = smiles_by_idx.loc[x_test_exp.index]
    candidate_test_idx = test_smiles[~test_smiles.isin(selected_molecules)].index.to_list()

    # If no valid candidate exists, just move train selected -> test (test set grows)
    if len(candidate_test_idx) == 0:
        x_test_exp = pd.concat([x_test_exp, x_train_exp.loc[[idx_train_sel]]], axis=0)
        y_test_exp = pd.concat([y_test_exp, y_train_exp.loc[[idx_train_sel]]], axis=0)

        x_train_exp = x_train_exp.drop(index=idx_train_sel)
        y_train_exp = y_train_exp.drop(index=idx_train_sel)
        continue

    # Choose one random test row to swap back into train
    idx_test_swap = rng.choice(candidate_test_idx)

    # Move selected train row -> test
    x_test_exp = pd.concat([x_test_exp, x_train_exp.loc[[idx_train_sel]]], axis=0)
    y_test_exp = pd.concat([y_test_exp, y_train_exp.loc[[idx_train_sel]]], axis=0)

    # Move random test row -> train
    x_train_exp = pd.concat([x_train_exp, x_test_exp.loc[[idx_test_swap]]], axis=0)
    y_train_exp = pd.concat([y_train_exp, y_test_exp.loc[[idx_test_swap]]], axis=0)

    # Remove original positions after swap
    x_train_exp = x_train_exp.drop(index=idx_train_sel)
    y_train_exp = y_train_exp.drop(index=idx_train_sel)
    x_test_exp = x_test_exp.drop(index=idx_test_swap)
    y_test_exp = y_test_exp.drop(index=idx_test_swap)

    swaps_done += 1

# Optional safety: remove accidental duplicate indices
x_train_exp = x_train_exp[~x_train_exp.index.duplicated(keep="first")]
y_train_exp = y_train_exp.loc[x_train_exp.index]
x_test_exp = x_test_exp[~x_test_exp.index.duplicated(keep="first")]
y_test_exp = y_test_exp.loc[x_test_exp.index]

# Check result
test_selected_count = smiles_by_idx.loc[x_test_exp.index].isin(selected_molecules).sum()
print(f"Swaps done: {swaps_done}")
print(f"Selected molecules now in test: {test_selected_count}/{len(selected_molecules)}")
print(f"Train size: {len(x_train_exp)}")
print(f"Test size: {len(x_test_exp)}")

In [ ]:
# get the full data for the entire test set
exp_test_indices = x_test_exp.index
exp_test_df = nmrshift_13C_1H_feat.loc[exp_test_indices].copy()
exp_train_df = nmrshift_13C_1H_feat.drop(exp_test_indices).copy()

## NMRShiftDB2 Chemical Space Plots

In [ ]:
# add features to visualize the chemical space covered by the molecules
nmrshift_chem_space = nmrshift_13C_1H_feat.copy()
# use rdkit for common values
nmrshift_chem_space['mw'] = nmrshift_chem_space['smiles'].apply(lambda x: Descriptors.MolWt(Chem.MolFromSmiles(x)))
nmrshift_chem_space['crippen_logp'] = nmrshift_chem_space['smiles'].apply(lambda x: Chem.Crippen.MolLogP(Chem.MolFromSmiles(x)))
nmrshift_chem_space['crippen_molmr'] = nmrshift_chem_space['smiles'].apply(lambda x: Chem.Crippen.MolMR(Chem.MolFromSmiles(x)))
nmrshift_chem_space['num_atom'] = nmrshift_chem_space['smiles'].apply(lambda x: Chem.rdMolDescriptors.CalcNumAtoms(Chem.MolFromSmiles(x)))

In [ ]:
# calc mol volume separately
for idx, row in nmrshift_chem_space.iterrows():
    mol = Chem.MolFromSmiles(row['smiles'])
    mol_with_hs = Chem.AddHs(mol) # add hydrogen for volume

    params = AllChem.ETKDGv3()
    params.randomSeed = random_seed
    embedded = False
    max_embed_tries = 3

    for attempt in range(max_embed_tries):
        mol_with_hs.RemoveAllConformers()
        status = AllChem.EmbedMolecule(mol_with_hs, params)

        if status == 0 and mol_with_hs.GetNumConformers() > 0:
            embedded = True
            break

        # relax settings for retry
        params.useRandomCoords = True
        params.randomSeed = random_seed + attempt + 1

    if not embedded:
        nmrshift_chem_space.at[idx, 'total_volume'] = np.nan
        continue

    dclv = rdMolDescriptors.DoubleCubicLatticeVolume(mol_with_hs)
    total_volume = dclv.GetVolume()
    vdw_volume = dclv.GetVDWVolume()
    polar_surface_area = rdMolDescriptors.CalcTPSA(mol_with_hs, includeSandP=True)
    surface_area = dclv.GetSurfaceArea()
    
    nmrshift_chem_space.at[idx, 'total_volume'] = total_volume
    nmrshift_chem_space.at[idx, 'vdw_volume'] = vdw_volume
    nmrshift_chem_space.at[idx, 'surface_area'] = surface_area
    nmrshift_chem_space.at[idx, 'polar_surface_area'] = polar_surface_area

In [ ]:
# rename variables for plotting
nmrshift_chem_space = nmrshift_chem_space.rename(columns={
    'mw': 'Molecular weight (Da)',
    'total_volume': 'Molecular volume (Å³)',
    'vdw_volume': 'Van der Waals volume (Å³)',
    'surface_area': 'Molecular surface area (Å²)',
    'polar_surface_area': 'Polar surface area (Å²)',
    'num_atom': 'Number of atoms',
    'NumHAcceptors': 'Number of hydrogen bond acceptors',
    'NumHDonors': 'Number of hydrogen bond donors',
    'NumRotatableBonds': 'Number of rotatable bonds',
    'NumHeteroatoms': 'Number of heteroatoms',
    'RingCount': 'Number of rings',
    'NumAliphaticRings': 'Number of aliphatic rings',
    'NumAromaticRings': 'Number of aromatic rings',
    'crippen_logp': 'Octanol-water partition coefficient (Crippen)',
    'crippen_molmr': 'Molar refractivity (Crippen)'
})

nmrshift_chem_space_vars = [
    'Molecular weight (Da)',
    'Molecular volume (Å³)',
    'Van der Waals volume (Å³)',
    'Molecular surface area (Å²)',
    'Polar surface area (Å²)',
    'Number of atoms',
    'Number of hydrogen bond acceptors',
    'Number of hydrogen bond donors',
    'Number of rotatable bonds',
    'Number of heteroatoms',
    'Number of rings',
    'Number of aliphatic rings',
    'Number of aromatic rings',
    'Octanol-water partition coefficient (Crippen)',
    'Molar refractivity (Crippen)'
]

In [ ]:
# Settings
df_plot = nmrshift_chem_space
vars_to_plot = nmrshift_chem_space_vars

n_cols = 3
n_rows = math.ceil(len(vars_to_plot) / n_cols)

sns.set_style("whitegrid")
mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = ["Nimbus Roman"]
mpl.rcParams["mathtext.fontset"] = "stix"
plt.rcParams['font.size'] = 12
fig = plt.figure(figsize=(5.2 * n_cols, 2.8 * n_rows), dpi=dpi_for_pub)
outer = fig.add_gridspec(n_rows, n_cols, wspace=0.30, hspace=0.50)

for i, var in enumerate(vars_to_plot):
    r, c = divmod(i, n_cols)

    # Each panel has 2 stacked axes: top boxplot, bottom density
    inner = outer[r, c].subgridspec(2, 1, height_ratios=[1, 4], hspace=0.05)
    ax_box = fig.add_subplot(inner[0])
    ax_den = fig.add_subplot(inner[1], sharex=ax_box)

    x = pd.to_numeric(df_plot[var], errors="coerce").dropna()

    if len(x) == 0:
        ax_den.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax_den.transAxes)        
        ax_box.axis("off")
        continue

    # Top: horizontal box-and-whisker
    sns.boxplot(
        x=x,
        ax=ax_box,
        orient="h",
        color="#c7d8f5",
        fliersize=2,
        linewidth=1
    )
    ax_box.set(yticks=[], ylabel="", xlabel="")
    ax_box.tick_params(axis="x", labelbottom=False)
    ax_box.set_title("")
    for spine in ["left", "right", "top"]:
        ax_box.spines[spine].set_visible(False)

    # Bottom: density plot
    # KDE needs more than one unique value; fallback to histogram if constant
    if x.nunique() > 1:
        sns.kdeplot(x=x, ax=ax_den, fill=True, color="#2a6fbb", linewidth=1.4)
    else:
        sns.histplot(x=x, ax=ax_den, bins=15, color="#2a6fbb")

    ax_den.set_ylabel("Density")
    ax_den.set_xlabel(var)

# Turn off any unused panels
total_slots = n_rows * n_cols
for j in range(len(vars_to_plot), total_slots):
    r, c = divmod(j, n_cols)
    inner = outer[r, c].subgridspec(2, 1, height_ratios=[1, 4], hspace=0.05)
    fig.add_subplot(inner[0]).axis("off")
    fig.add_subplot(inner[1]).axis("off")

plots_dir = os.path.join(os.getcwd(), "plots")
plt.savefig(os.path.join(plots_dir, "figure s11.png"), dpi=dpi_for_pub, bbox_inches='tight')
plt.tight_layout()
plt.show()

## NMRShiftDB2 Hyperparameter Optimization

In [ ]:
# set up k-fold cv to use inside Optuna
cv = 10 # 10-fold cross-validation
kf = KFold(n_splits=cv, shuffle=True, random_state=random_seed)

# create precomputed folds to speed up loop
precomputed_folds_nmrshift = [
    (train_idx.astype(np.int32), test_idx.astype(np.int32))
    for train_idx, test_idx in kf.split(exp_train_df)
]
print(f"Precomputed {len(precomputed_folds_nmrshift)} folds")

In [ ]:
# Build simple mlp
def set_global_seed(seed: int = random_seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    # torch.backends.cudnn.deterministic = True T not used due to slowdown of MLP
    torch.backends.cudnn.benchmark = True

# specify possible activation functions
_ACTS = {
    "relu": nn.ReLU,
    "elu": nn.ELU,
    "gelu": nn.GELU,
    "leaky_relu": nn.LeakyReLU,
    "tanh": nn.Tanh,
    "sigmoid": nn.Sigmoid
}

class MLPRegressorModule(nn.Module):
    def __init__(
        self,
        input_dim: int,
        hidden_layer_sizes=(8, 6, 4), # inital layer sizes
        activation="relu", # base case with relu
        dropout=0.0, # no dropout for base
    ):
        super().__init__()
        if activation not in _ACTS:
            raise ValueError(f"activation must be one of {list(_ACTS.keys())}")

        act = _ACTS[activation]
        h1, h2, h3 = hidden_layer_sizes

        self.net = nn.Sequential(
            nn.Linear(input_dim, h1),
            act(),
            nn.Dropout(dropout),

            nn.Linear(h1, h2),
            act(),
            nn.Dropout(dropout),

            nn.Linear(h2, h3),
            act(),
            nn.Dropout(dropout),

            nn.Linear(h3, 1),
        )

    def forward(self, X):
        return self.net(X)
        
# build new default MLP
def make_mlp_skorch_estimator_nmrshift(input_dim: int):
    net = NeuralNetRegressor(
        module=MLPRegressorModule,
        module__input_dim=input_dim,
        module__hidden_layer_sizes=(8, 4, 2),
        module__activation="relu",
        module__dropout=0.0,

        criterion=nn.MSELoss, # only training by MSE
        optimizer=torch.optim.Adam, # only using Adam

        lr=1e-3,
        max_epochs=100,
        batch_size=128,

        device="cuda" if torch.cuda.is_available() else "cpu",

        # we're enabling validation splits inside so we can aggressively monitor validation loss
        train_split=ValidSplit(cv=0.2, stratified=False, random_state=random_seed), # upped cv from 0.1 to 0.2 given smaller sizes

        callbacks=[
            EarlyStopping(monitor="valid_loss", patience=8, threshold=1e-3, lower_is_better=True),
        ],

        verbose=0,
    )
    return net

In [ ]:
# implement hpo search using optuna
# here we use different studies per model family so that we have the same number of trials for each model family

def suggest_params_nmrshift(trial, model_name, input_dim=None):
    # Return a dict of params to pass into model.set_params(**params)
    if model_name == "MLR_nmrshift":
        return {}
        
    if model_name == "Lasso_nmrshift":
        return {
            'alpha': trial.suggest_float('alpha', 1e-5, 1e1, log=True),
            'tol': trial.suggest_float('tol', 1e-6, 1e-2, log=True)
    }

    if model_name == "KNN_nmrshift":
        return {
            'n_neighbors': trial.suggest_int('n_neighbors', 1, 75, log=True),
            'metric': trial.suggest_categorical('metric', ['euclidean', 'manhattan']),
            'weights': trial.suggest_categorical('weights', ['uniform', 'distance'])
        }

    if model_name == 'SVR_nmrshift':
        return {
            'max_iter': trial.suggest_int('max_iter', 200, 5000, log=True),
            'kernel': trial.suggest_categorical('kernel', ['linear', 'rbf']),
            'C': trial.suggest_float('C', 1e-2, 1e3, log=True),
            'epsilon': trial.suggest_float('epsilon', 1e-3, 1.0, log=True),
            'gamma': trial.suggest_float('gamma', 1e-4, 1e1, log=True) # note that this only applies to the rbf kernel
        }
    if model_name == 'RandomForest_nmrshift':
        return {
            'n_estimators': trial.suggest_int('n_estimators', 100, 1200),
            'split_criterion': trial.suggest_categorical('split_criterion', ['mse', 'poisson', 'gamma', 'inverse_gaussian']), # all but mse have requirements for y target values
            'max_depth': trial.suggest_int('max_depth', 3, 15),
            'max_features': trial.suggest_categorical('max_features', ['log2', 'sqrt', 0.3, 0.5, 1.0]),
            'min_samples_leaf': trial.suggest_int('min_samples_leaf', 2, 40),
            'min_samples_split': trial.suggest_int('min_samples_split', 4, 80),
            'bootstrap': trial.suggest_categorical('bootstrap', [True, False])
        }
    if model_name == 'XGBoost_nmrshift':
        return {
            'max_depth':  trial.suggest_int('max_depth', 2, 6),
            'eta': trial.suggest_float('eta', 1e-2, 3e-1, log=True),
            'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
            'subsample': trial.suggest_float('subsample', 0.4, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
            'min_child_weight': trial.suggest_float('min_child_weight', 1e-2, 20.0, log=True),
            'gamma': trial.suggest_float('gamma', 1e-4, 5.0, log=True),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 1.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 100.0, log=True),
        }
    if model_name == "MLP_skorch_nmrshift":
        seed = trial.suggest_categorical('seed', [random_seed]) # pin to random seed
        h1 = trial.suggest_int("h1", 8, 32, step=4)
        h2 = trial.suggest_int("h2", 4, 16, step=2)
        h3 = trial.suggest_int("h3", 2, 8, step=1)

        return {
            # skorch top-level params
            "seed": seed,
            "device": trial.suggest_categorical("device", ["cuda"]),  # only run on GPU
            "max_epochs": trial.suggest_int("max_epochs", 50, 225, step=25),
            "batch_size": trial.suggest_categorical("batch_size", [32, 64, 128, 256, 512]),
            "lr": trial.suggest_float("lr", 5e-4, 2e-1, log=True), # this samples from log distribution

            # optimizer params
            "optimizer__weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True),

            # module params
            "module__input_dim": input_dim,
            "module__hidden_layer_sizes": (h1, h2, h3),
            "module__activation": trial.suggest_categorical("activation", ["relu", "elu", "gelu", "leaky_relu", "tanh", "sigmoid"]),
            "module__dropout": trial.suggest_float("dropout", 1e-6, 0.5),

            "verbose": 0,
        }
        
    return {}

In [ ]:
# List of dictionaries of models for hyperparameter tuning
gpu_models_and_params_nmrshift = [
    {'name': 'MLR_nmrshift', 'estimator': cuml_mlr()},
    {'name': 'Lasso_nmrshift', 'estimator': cuml_lasso(
        max_iter = 20000
    )},
    {'name': 'KNN_nmrshift', 'estimator': cuml_knn()},
    {'name': 'SVR_nmrshift', 'estimator': cuml_svr()},
    {'name': 'RandomForest_nmrshift', 'estimator': cuml_rf(
        random_state = random_seed
    )},
    {'name': 'XGBoost_nmrshift', 'estimator': XGBRegressor(
        random_state = random_seed,
        tree_method = 'hist',
        device = 'cuda'
    )},
    {'name': 'MLP_skorch_nmrshift', 'estimator': make_mlp_skorch_estimator_nmrshift(input_dim=x_train_exp.shape[1])}
]

In [ ]:
def make_objective_nmrshift(model_name, base_estimator, x_cv, y_cv, folds, results_store, property_name="HOMO-LUMO Gap"):
    def objective_nmrshift(trial):
        # make a "base/default" evaluation as trial 0        
        if trial.number == 0:
            params = {}
        else:
            params = suggest_params_nmrshift(trial, model_name, input_dim=x_cv.shape[1])
        seed = None
        if model_name == "MLP_skorch_nmrshift" and params and "seed" in params:
            params = dict(params)
            seed = int(params.pop("seed"))
            
        model = clone(base_estimator)
        if params:
            model.set_params(**params)

        run_label = f"{model_name}__base" if not params else f"{model_name}__{params}"
        if model_name == "MLP_skorch_nmrshift" and seed is not None:
            run_label += f"__seed={seed}"
                
        mse_folds, rmse_folds, mae_folds, q2_folds = [], [], [], []

        for fold_idx, (train_idx, test_idx) in enumerate(folds, start=1):
            x_tr, x_te = x_cv.iloc[train_idx], x_cv.iloc[test_idx]
            y_tr, y_te = y_cv.iloc[train_idx], y_cv.iloc[test_idx]

            # tree-based not need scaling
            if model_name in ("RandomForest_nmrshift", "XGBoost_nmrshift"):
                x_tr_norm, x_te_norm = x_tr, x_te
            else: # Standard scaling on train applied to test
                scaler = StandardScaler()
                x_tr_norm = scaler.fit_transform(x_tr)
                x_te_norm = scaler.transform(x_te)
                
            try:
                if model_name == "XGBoost_nmrshift":
                    x_tr_gpu = cudf.from_pandas(x_tr_norm)
                    x_te_gpu = cudf.from_pandas(x_te_norm)
                    y_tr_gpu = cudf.Series(y_tr.values)
                    model.fit(x_tr_gpu, y_tr_gpu)
                    y_pred = model.predict(x_te_gpu)
                elif model_name == "MLP_skorch_nmrshift":
                    if seed is not None:
                        set_global_seed(int(random_seed))

                    # skorch expects numpy float32; y typically as (N, 1)
                    x_tr32 = x_tr_norm.astype(np.float32, copy=False)
                    x_te32 = x_te_norm.astype(np.float32, copy=False)
                    y_tr32 = y_tr.to_numpy(dtype=np.float32).reshape(-1, 1)

                    t0 = time.time()
                    model.fit(x_tr32, y_tr32)
                    device_used = next(model.module_.parameters()).device
                    n_epochs_run = len(model.history)
                    print(f"[{device_used}] fit took {time.time() - t0:.1f}s over {n_epochs_run} epochs")
                    assert device_used.type == "cuda", "MLP fell back to CPU!"
                    y_pred = model.predict(x_te32).reshape(-1)
                else:
                    model.fit(x_tr_norm, y_tr)
                    y_pred = model.predict(x_te_norm)

                # metrics for saving and pruning
                fold_mse = mean_squared_error(y_te, y_pred)
                mse_folds.append(mean_squared_error(y_te, y_pred))
                rmse_folds.append(root_mean_squared_error(y_te, y_pred))
                mae_folds.append(mean_absolute_error(y_te, y_pred))
                q2_folds.append(r2_score(y_te, y_pred))

                # pruning
                running_mse = float(np.mean(mse_folds))
                trial.report(running_mse, step=fold_idx)  # step == CV fold count

                if trial.should_prune():
                    raise optuna.exceptions.TrialPruned()

            except optuna.exceptions.TrialPruned:
                # IMPORTANT: don't swallow pruning
                raise
            except Exception as e:
                print(f"Fold error for {run_label}: {e}")
                continue

        # If all folds failed, penalize the trial
        if len(rmse_folds) == 0:
            return float("inf")

        row = {
            "Property": property_name,
            "Model": run_label,
            "trial_number": trial.number,
            "MSE_mean": float(np.mean(mse_folds)),
            "MSE_std": float(np.std(mse_folds)),
            "RMSE_mean": float(np.mean(rmse_folds)),
            "RMSE_std": float(np.std(rmse_folds)),
            "MAE_mean": float(np.mean(mae_folds)),
            "MAE_std": float(np.std(mae_folds)),
            "Q2_mean": float(np.mean(q2_folds)),
            "Q2_std": float(np.std(q2_folds)),
            "n_folds_used": len(rmse_folds),
            "model_name": model_name,
        }
        
        # Persist metrics into Optuna storage (SQLite) for this trial
        for k in ["MSE_mean","MSE_std","RMSE_mean","RMSE_std","MAE_mean","MAE_std","Q2_mean","Q2_std","n_folds_used"]:
            trial.set_user_attr(k, row[k])
        
        # Optional: store a compact label / params used
        trial.set_user_attr("run_label", run_label)
        trial.set_user_attr("model_name", model_name)
        trial.set_user_attr("Property", property_name)

        # Optimize MSE (minimize)
        return row["MSE_mean"]

    return objective_nmrshift

In [ ]:
# Set up for persistent storage
runs_dir = os.path.join(os.getcwd(), "hpo")
os.makedirs(runs_dir, exist_ok=True)

db_path = os.path.join(runs_dir, "optuna_nmrshift_hpo.db")

# Note: sqlite://// is correct for an absolute Linux path.
OPTUNA_STORAGE_nmrshift = f"sqlite:////{db_path}"

# Optional: include property name so you don't mix studies across targets
PROPERTY_NAME = "HOMO-LUMO Gap"
STUDY_PREFIX = f"MSE__{PROPERTY_NAME}"

# add a pruner
pruner = optuna.pruners.MedianPruner(
    n_startup_trials=20,   # don’t prune the first few trials
    n_warmup_steps=4,     # don’t prune before cv fold 4
    interval_steps=1
)

def get_or_create_study(model_name: str):
    return optuna.create_study(
        direction="minimize",
        study_name=f"{STUDY_PREFIX}__{model_name}",
        storage=OPTUNA_STORAGE_nmrshift,
        load_if_exists=True,
        pruner=pruner # adds pruning
    )

# Build/load all studies once
studies_nmrshift = {m["name"]: get_or_create_study(m["name"]) for m in gpu_models_and_params_nmrshift}

# Quick check
for name, st in studies_nmrshift.items():
    print(name, "existing trials:", len(st.trials))

In [ ]:
results_nmrshift = []  # if you want *results across this session*
N_TRIALS = 200

def run_hpo_for_model_nmrshift(model_name: str, n_trials: int):
    model_spec = next(m for m in gpu_models_and_params_nmrshift if m["name"] == model_name)
    base_estimator = model_spec["estimator"]

    study = studies_nmrshift[model_name]  # already persistent

    objective_nmrshift = make_objective_nmrshift(
        model_name=model_name,
        base_estimator=base_estimator,
        x_cv=x_train_exp,
        y_cv=y_train_exp,
        folds=precomputed_folds_nmrshift,
        results_store=results_nmrshift,
        property_name=PROPERTY_NAME
    )

    study.optimize(objective_nmrshift, n_trials=n_trials)
    return study

# Run all models (like your current loop), but now resumable:
for model_spec in tqdm(gpu_models_and_params_nmrshift, desc="Model loop"):
    model_name = model_spec["name"]
    n_trials = 1 if model_name == "MLR_nmrshift" else N_TRIALS
    run_hpo_for_model_nmrshift(model_name, n_trials=n_trials)

In [ ]:
def best_params_str(study):
    bp = study.best_trial.params
    return "base/default" if len(bp) == 0 else ", ".join(f"{k}={v}" for k, v in bp.items())

nmrshift_rows = []
for m, st in studies_nmrshift.items():
    bt = st.best_trial
    ua = bt.user_attrs

    nmrshift_rows.append({
        "model_name": m,
        "MSE_mean": ua.get("MSE_mean", float(bt.value)),
        "MSE_std": ua.get("MSE_std", np.nan),
        "RMSE_mean": ua.get("RMSE_mean", np.nan),
        "RMSE_std": ua.get("RMSE_std", np.nan),
        "MAE_mean": ua.get("MAE_mean", np.nan),
        "MAE_std": ua.get("MAE_std", np.nan),
        "Q2_mean": ua.get("Q2_mean", np.nan),
        "Q2_std": ua.get("Q2_std", np.nan),
        "n_folds_used": ua.get("n_folds_used", np.nan),
        "best_params_str": best_params_str(st),
        "best_trial_number": bt.number,
        "n_trials_total": len(st.trials),
    })

nmrshift_leaderboard = pd.DataFrame(nmrshift_rows).sort_values("MSE_mean").reset_index(drop=True)
nmrshift_leaderboard

In [ ]:
# Save HPO results table to CSV
os.chdir("hpo")
nmrshift_leaderboard.to_csv("nmrshift_leaderboard.csv", index=False)
print("Saved NMRShiftDB2 HPO results.csv")
os.chdir("..")

In [ ]:
nmrshift_leaderboard[['model_name', 'best_params_str']].iloc[5,1]

## NMRShiftDB2 Evaluation

In [ ]:
# starting with base models,
base_models = {
    "MLR_nmrshift": cuml_mlr(),
    "Lasso_nmrshift": cuml_lasso(),
    "KNN_nmrshift": cuml_knn(),
    "SVR_nmrshift": cuml_svr(),
    "RandomForest_nmrshift": cuml_rf(),
    "XGBoost_nmrshift": XGBRegressor(),
    "MLP_skorch_nmrshift": make_mlp_skorch_estimator_nmrshift(input_dim=x_train_exp.shape[1])
}

# create final fitted-ready estimators using best params from Optuna DB
final_models = {}
for name, base_est in base_models.items():
    est = clone(base_est)
        
    raw_best = studies_nmrshift[name].best_trial.params

    # Replay the same suggest_params logic with fixed values,
    # so h1/h2/h3 -> hidden_layer_sizes, weight_decay -> optimizer__weight_decay,
    # module__input_dim gets included, etc. all happen automatically.
    fixed_trial = optuna.trial.FixedTrial(raw_best)
    best_params = suggest_params_nmrshift(fixed_trial, name, input_dim=x_train_exp.shape[1])

    # seed isn't a real skorch/estimator param, still needs dropping
    best_params = {k: v for k, v in best_params.items() if k not in {"seed"}}

    if best_params:
        est.set_params(**best_params)
    final_models[name] = est

final_models_and_params_nmrshift = [{"name": k, "estimator": v} for k, v in final_models.items()]

In [ ]:
# train and test best models
results_nmrshiftdb2  = []

# Z-score normalization (fit only on training set)
scaler = StandardScaler()
X_train_scaled_exp = scaler.fit_transform(x_train_exp).astype(np.float32)
X_test_scaled_exp = scaler.transform(x_test_exp).astype(np.float32)

for model_info in tqdm(final_models_and_params_nmrshift):
    name = model_info['name']
    estimator = model_info['estimator']

    if name == "MLP_skorch":
        y_train_exp = y_train_exp.to_numpy(dtype=np.float32).reshape(-1, 1)
    else:
        y_train_exp = y_train_exp
        
    # Fit on training set
    estimator.fit(X_train_scaled_exp, y_train_exp)
    # Predict on test set
    y_pred_exp = estimator.predict(X_test_scaled_exp)
    if name == "MLP_skorch":
        y_pred_exp = y_pred_exp.reshape(-1)
    # Metrics
    mse = mean_squared_error(y_test_exp, y_pred_exp)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test_exp, y_pred_exp)
    r2 = r2_score(y_test_exp, y_pred_exp)
    results_nmrshiftdb2 .append({
        'Model': name,
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2
        
    })

test_results_df_nmrshiftdb2 = pd.DataFrame(results_nmrshiftdb2)

In [ ]:
test_results_df_nmrshiftdb2.head(7)

In [ ]:
# gpu based, but this doesn't work for SHAP given model size
nmrshift_estimator_gpu = next(d["estimator"] for d in final_models_and_params_nmrshift if d["name"] == "XGBoost_nmrshift")

# gpu vs cpu selection
nmrshit_estimator = nmrshift_estimator_gpu

In [ ]:
exp_results = []

# Standardize features using training set stats
scaler = StandardScaler()
x_train_exp_scaled = scaler.fit_transform(x_train_exp).astype(np.float32)
x_test_exp_scaled = scaler.transform(x_test_exp).astype(np.float32)

# Train model
nmrshit_estimator.fit(x_train_exp_scaled, y_train_exp)

# Predict and evaluate
y_pred = nmrshit_estimator.predict(x_test_exp_scaled)
mae = mean_absolute_error(y_test_exp, y_pred)
rmse = np.sqrt(mean_squared_error(y_test_exp, y_pred))
r2 = r2_score(y_test_exp, y_pred)

# Add columns to test_df
exp_test_df["Predicted"] = y_pred
exp_test_df["Actual"] = y_test_exp
exp_test_df["Residual"] = y_test_exp - y_pred

# Store results
exp_results.append({
    "MAE": mae,
    "RMSE": rmse,
    "R2": r2
})

# Output results table
exp_results_df = pd.DataFrame(exp_results)
exp_results_df

In [ ]:
# get feature importances for best model
nmrshift_importances = nmrshit_estimator.feature_importances_

In [ ]:
# parallelize to speed up SHAP
# for smaller set, do all SHAP
X_subset = x_train_exp_scaled[0:len(x_train_exp_scaled)]

def explain_chunk(model, X_chunk):
    explainer = shap.TreeExplainer(model, feature_perturbation="tree_path_dependent")
    values = explainer.shap_values(X_chunk, check_additivity=True)
    base_value = explainer.expected_value
    return values, base_value

n_jobs = 32
chunks = np.array_split(X_subset, n_jobs)

nmrshift_results = Parallel(n_jobs=n_jobs, backend="loky")(
    delayed(explain_chunk)(nmrshit_estimator, chunk) for chunk in chunks
)

# Reassemble
shap_values_list, base_values_list = zip(*nmrshift_results)
shap_values_full = np.concatenate(shap_values_list, axis=0)

# base_value is a scalar (or array of 1 for regression) - same across chunks, just grab one
base_value = base_values_list[0]
if isinstance(base_value, np.ndarray):
    base_value = base_value.item() if base_value.size == 1 else base_value

# Build base_values array matching number of rows (Explanation expects one per row)
base_values_full = np.full(shap_values_full.shape[0], base_value)

# Build the full Explanation object, same shape as before
nmrshift_explanation = shap.Explanation(
    values=shap_values_full,
    base_values=base_values_full,
    data=X_subset,
    feature_names=features
)

## NMRShiftDB2 Plotting

In [ ]:
def _non_constant_params(study: optuna.study.Study):
    complete = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    params = sorted({k for t in complete for k in t.params.keys()})
    keep, dropped = [], []
    for p in params:
        vals = [t.params[p] for t in complete if p in t.params]
        (keep if len(set(vals)) >= 2 else dropped).append(p)
    return keep, dropped
    
def plot_global_importances_2perrow_nmrshift(
    studies,                 # dict: {model_name: study}
    top_n: int | None = 15,
    figsize=(8, 4),          # per-plot size (w, h)
    drop_constants: bool = True,
    evaluator=None,
    show_dropped: bool = True,
    dpi=None,                # e.g. dpi_for_pub
    skip_names=("MLR_nmrshift",),
):
    items = [(name, st) for name, st in studies.items() if name not in set(skip_names)]
    n = len(items)
    ncols = 2
    nrows = math.ceil(n / ncols)

    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(figsize[0] * ncols, figsize[1] * nrows),
        dpi=dpi,
        squeeze=False
    )

    for i, (model_name, study) in enumerate(items):
        r, c = divmod(i, ncols)
        ax = axes[r, c]

        # --- BEGIN: your existing code, with ONLY "ax=" substitutions and no plt.show() ---
        complete = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
        if not any(t.params for t in complete):
            ax.axis("off")
            ax.text(0.0, 0.6, f"{model_name} ({len(study.trials)} trials)\n(no COMPLETE trials with params)", fontsize=12)
            continue

        if evaluator is None:
            _evaluator = FanovaImportanceEvaluator(seed=0)
        else:
            _evaluator = evaluator

        if drop_constants:
            params_keep, params_dropped = _non_constant_params(study)
        else:
            params_keep = None
            params_dropped = []

        try:
            imp = get_param_importances(study, evaluator=_evaluator, params=params_keep)
        except Exception as e:
            ax.axis("off")
            ax.text(
                0.0, 0.6,
                f"{model_name} ({len(study.trials)} trials)\n"
                f"importance computation failed:\n{type(e).__name__}: {e}",
                fontsize=11
            )
            continue

        if not imp:
            ax.axis("off")
            msg = f"{model_name} ({len(study.trials)} trials)\n(no importances returned)"
            if drop_constants and show_dropped and params_dropped:
                msg += f"\n(dropped constants: {', '.join(params_dropped)})"
            ax.text(0.0, 0.6, msg, fontsize=11)
            continue

        items2 = list(imp.items())
        if top_n is not None:
            items2 = items2[:top_n]

        labels = [k for k, _ in items2][::-1]
        vals = [v for _, v in items2][::-1]

        ax.barh(labels, vals, color="0.6")
        ax.set_xlabel("Importance")
        ax.set_title(f"{model_name} (200 trials) — Global hyperparameter importances") # three MLP failures means there were technically 103 MLP trials
        ax.set_xlim(0, max(vals) * 1.1)

        if drop_constants and show_dropped and params_dropped:
            ax.text(
                1.0, -0.12,
                f"Dropped constant params: {', '.join(params_dropped)}",
                transform=ax.transAxes,
                ha="right", va="top",
                fontsize=9
            )
        # --- END: your existing code ---

    # turn off any unused subplot(s)
    for j in range(n, nrows * ncols):
        r, c = divmod(j, ncols)
        axes[r, c].axis("off")

    plt.tight_layout()
    plots_dir = os.path.join(os.getcwd(), "plots")
    plt.savefig(os.path.join(plots_dir, "figure s13.png"), dpi=dpi_for_pub, bbox_inches='tight')
    plt.show()

In [ ]:
plot_global_importances_2perrow_nmrshift(
    studies_nmrshift,
    top_n=10,
    figsize=(8, 4),
    dpi=dpi_for_pub,
)

In [ ]:
# create solvent labels for specific groups of solvents
# make this a new column in test_df
chloroform_names = [
    'Chloroform-D1 (CDCl3)', 'CDCl3', 'CDCL3,TMS'
]
known_names = [
    'Deuteriumoxide (D2O)',
    'Methanol-D4 (CD3OD)',  'D2O',
    'Dimethylsulphoxide-D6 (DMSO-D6, C2D6SO)', 'Pyridin-D5 (C5D5N)',
    'Acetone-D6 ((CD3)2CO)', 'Acetonitrile-D3 (CD3CN)', 'DMSO', 'acetone',
    'benzene', 'Benzene-D6 (C6D6)', 'Toluol-d8'
]
unknown_names = [
    'Unreported'
]

def label_solvent(solvent):
    if solvent in chloroform_names:
        return 'Deuterated\nchloroform'
    elif solvent in known_names:
        return 'Other\nsolvent'
    elif solvent in unknown_names:
        return 'Unknown\nsolvent'

# Apply function to create new column
exp_test_df['Solvent_Group'] = exp_test_df['Solvent'].apply(label_solvent)

In [ ]:
# bool functions for atom flags
def is_chno_only(formula):
    # Matches only C, H, N, O (with optional numbers), nothing else
    return bool(re.fullmatch(r'(C\d*|H\d*|N\d*|O\d*)+', formula))

def is_chno_with_halogen(formula):
    # Must contain only C, H, N, O, F, Cl, Br, I (with optional numbers)
    # And must contain at least one halogen: F, Cl, Br, I
    allowed = re.fullmatch(r'(C\d*|H\d*|N\d*|O\d*|F\d*|Cl\d*|Br\d*|I\d*)+', formula)
    has_halogen = any(x in formula for x in ['F', 'Cl', 'Br', 'I'])
    return bool(allowed and has_halogen)

def contains_s_p_b(formula):
    return 'S' in formula or 'P' in formula or 'B' in formula

# add molecules flags
exp_test_df['molecular_formula'] = exp_test_df['molecular_formula'].str.replace('+', '', regex=False) # remove "+" which causes problems
exp_test_df['molecular_formula'] = exp_test_df['molecular_formula'].str.replace('-', '', regex=False) # remove "-" which causes problems
exp_test_df["CHNO_only"] = exp_test_df["molecular_formula"].apply(is_chno_only)
exp_test_df["With_halogen"] = exp_test_df["molecular_formula"].apply(is_chno_with_halogen)
exp_test_df["With_SPB"] = exp_test_df["molecular_formula"].apply(contains_s_p_b)

In [ ]:
selected_molecules = [
    ('O=C1c2ccccc2C(=O)c2c1ccc(O)c2O'), #"Alizarin - red dye", 
    ('O=C(O)/C=C/c1ccc(O)cc1'), # "p-Coumaric acid - UV-active"
    ('CC(C)=C/C=C/C(C)=C/C=C/C(C)=C1\C(=O)C[C@H]2[C@@]3(C)CC[C@@H](O)[C@](C)(CO)[C@@H]3CC[C@]12C'), # Stellettin J
]

selected_molecules_df = exp_test_df[exp_test_df['smiles'].isin(selected_molecules)]
selected_molecules_df['labels'] = ['p-Coumaric acid\n(UV-active)',
                                   'Alizarin\n(Red dye)',
                                   'Stellettin J\n(Yellow pigment)'].copy()

In [ ]:
selected_molecules_df

### figure 5

In [ ]:
# overall parameters
mpl.rcParams['hatch.linewidth'] = 1.2 # Specify hatch line width (increase for denser/darker hatch)
mpl.rcParams['hatch.color'] = 'black'  # hatch color (optional)

# set figure size
fig = plt.figure(figsize=(14, 8), dpi=dpi_for_pub)
gs = gridspec.GridSpec(2, 2, width_ratios=[2, 1], height_ratios=[2, 1], wspace=0.1, hspace=0.2)


# Panel A: predicted vs actual, MW with atoms as shading
x = exp_test_df['Actual'].values
y = exp_test_df['Predicted'].values
mol_weight = exp_test_df['mol_weight'].values
ax_left = fig.add_subplot(gs[:, 0])

is_chno = exp_test_df["CHNO_only"].astype(bool).values
is_halogen = exp_test_df["With_halogen"].astype(bool).values
is_spb = exp_test_df["With_SPB"].astype(bool).values

size = 40
linewidth = 0.4
alpha = 0.7

# Plot CHNO only: solid face, no hatch
sc1 = ax_left.scatter(x[is_chno], y[is_chno], c=mol_weight[is_chno], cmap='spring', s=size, alpha=alpha,
                  edgecolors='black', linewidths=linewidth, marker='o', label='C, H, N, O only', zorder=1)
# Plot halogen: hatched with lines
sc2 = ax_left.scatter(x[is_halogen], y[is_halogen], c=mol_weight[is_halogen], cmap='spring', s=size, alpha=alpha,
                  edgecolors='black', linewidths=linewidth, marker='o', label='+F, Cl, Br, I', zorder=2, hatch='////////')
# Plot SPB: hatched with dots
sc3 = ax_left.scatter(x[is_spb], y[is_spb], c=mol_weight[is_spb], cmap='spring', s=size, alpha=alpha,
                  edgecolors='black', linewidths=linewidth, marker='o', label='+S, P, B', zorder=3, hatch='.....')

# Example: manually specify label positions for each molecule
label_positions = {
    # 'smiles': (x_offset, y_offset)
    'CC(C)=C/C=C/C(C)=C/C=C/C(C)=C1\C(=O)C[C@H]2[C@@]3(C)CC[C@@H](O)[C@](C)(CO)[C@@H]3CC[C@]12C': (0.5, 3.5),
    'O=C1c2ccccc2C(=O)c2c1ccc(O)c2O': (0.4, 3.1),  # adjust offsets as needed
    'O=C(O)/C=C/c1ccc(O)cc1': (0.2, 1.4),
    # Add more entries for each molecule
}

for i, row in enumerate(selected_molecules_df.itertuples()):
    ax_left.scatter(row.Actual, row.Predicted, s=120, marker='*', color='#1e74d2', zorder=4, edgecolor='black')
    offset = label_positions.get(row.smiles, (0.15, 0.15))
    label_x = row.Actual + offset[0]
    label_y = row.Predicted + offset[1]
    ax_left.text(label_x, label_y, row.labels, fontsize=10, ha='center', va='bottom', color='black', fontweight='bold')
    ax_left.annotate(
        '', 
        xy=(row.Actual, row.Predicted), 
        xytext=(label_x, label_y),
        arrowprops=dict(arrowstyle='->', color='black', lw=1.5)
    )
    
# Add colorbar for mol_weight
cbar = plt.colorbar(sc1, ax=ax_left)
cbar.set_label('Molecular weight')

lo = min(x.min(), y.min())
hi = max(x.max(), y.max())
ax_left.plot([lo, hi], [lo, hi], 'r--', linewidth=1, zorder=0)
ax_left.set_xlabel('ωB97X-D/def2svp computed $\\Delta$E (eV)')
ax_left.set_ylabel('Experimental NMR-predicted $\\Delta$E (eV)')
ax_left.text(-0.05, 0.98, 'A', transform=ax_left.transAxes, fontsize=16, fontweight='bold', va='top', ha='right')

# Custom legend for shading
legend_handles = [
    mpatches.Patch(facecolor='white', edgecolor='black', label='C, H, N, O only'),
    mpatches.Patch(facecolor='white', edgecolor='black', hatch='////////', label='+F, Cl, Br, I'),
    mpatches.Patch(facecolor='white', edgecolor='black', hatch='.....', label='+S, P, B')
]
ax_left.legend(handles=legend_handles, title='Atom Group')

# add metrics
r2 = r2_score(exp_test_df['Actual'].values, exp_test_df['Predicted'].values)
mae = mean_absolute_error(exp_test_df['Actual'].values, exp_test_df['Predicted'].values)

# Atom group MAEs
mae_chno = mean_absolute_error(x[is_chno], y[is_chno])
mae_halogen = mean_absolute_error(x[is_halogen], y[is_halogen])
mae_spb = mean_absolute_error(x[is_spb], y[is_spb])

metrics_text = (
    f"R² = {r2:.3f}\n"
    f"MAE (all) = {mae:.3f}\n"
    f"MSE (all) = {mean_squared_error(exp_test_df['Actual'].values, exp_test_df['Predicted'].values):.3f}\n"
    f"RMSE (all) = {np.sqrt(mean_squared_error(exp_test_df['Actual'].values, exp_test_df['Predicted'].values)):.3f}\n"
    f"MAE (C, H, N, O) = {mae_chno:.3f}\n"
    f"MAE (+F, Cl, Br, I) = {mae_halogen:.3f}\n"
    f"MAE (+S, P, B) = {mae_spb:.3f}"
)
ax_left.text(
    0.98, 0.02, metrics_text,
    transform=ax_left.transAxes,
    fontsize=10,
    verticalalignment='bottom',
    horizontalalignment='right',
    bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='black', alpha=0.8)
)



# Top-right panel
ax_topright = fig.add_subplot(gs[0, 1])
x = exp_test_df['Actual'].values
y = exp_test_df['Predicted'].values
has_charge = exp_test_df['has_charge'].values if 'has_charge' in exp_test_df.columns else exp_test_df['smiles'].apply(lambda s: ('+' in s) or ('-' in s)).values
neutral_color = '#00a77e'
charged_color = '#d46200'
ax_topright.scatter(x[~has_charge], y[~has_charge], s=32, alpha=1, edgecolors='black', linewidths=0.4, color=neutral_color, zorder=1, label='Neutral')
ax_topright.scatter(x[has_charge], y[has_charge], s=32, alpha=1, edgecolors='black', linewidths=0.4, color=charged_color, zorder=2, label='Charged')
lo = min(x.min(), y.min())
hi = max(x.max(), y.max())
ax_topright.plot([lo, hi], [lo, hi], 'r--', linewidth=1, zorder=0)
ax_topright.set_xlabel('ωB97X-D/def2svp computed $\\Delta$E (eV)')
ax_topright.set_ylabel('Experimental NMR-predicted $\\Delta$E (eV)')
ax_topright.legend(title='Charge State', loc='upper left')
ax_topright.text(-0.12, 0.98, 'B', transform=ax_topright.transAxes, fontsize=16, fontweight='bold', va='top', ha='right')

# Charge MAEs
mae_charged = mean_absolute_error(x[has_charge], y[has_charge]) if has_charge.any() else float('nan')
mae_uncharged = mean_absolute_error(x[~has_charge], y[~has_charge]) if (~has_charge).any() else float('nan')

# add metrics to Panel B lower
metrics_text_b = (
    f"MAE (Charged) = {mae_charged:.3f}\n"
    f"MAE (Neutral) = {mae_uncharged:.3f}"
)
ax_topright.text(
    0.98, 0.02, metrics_text_b,
    transform=ax_topright.transAxes,
    fontsize=10,
    verticalalignment='bottom',
    horizontalalignment='right',
    bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='black', alpha=0.8)
)

# Bottom right panel
ax_bottomright = fig.add_subplot(gs[1, 1])

solvent_counts = exp_test_df['Solvent_Group'].value_counts().reindex(['Deuterated\nchloroform',
                                                                  'Other\nsolvent',
                                                                  'Unknown\nsolvent'], fill_value=0)
pie_colors = ['#6eda19', '#effe01', '#0dcbb7']
wedges, texts, autotexts = ax_bottomright.pie(solvent_counts, labels=solvent_counts.index, colors=pie_colors, autopct='%1.1f%%', startangle=90, textprops={'fontsize': 12})
ax_bottomright.text(-0.55, 0.98, 'C', transform=ax_bottomright.transAxes, fontsize=16, fontweight='bold', va='top', ha='right')

# Add a box around the pie chart
rect = mpatches.Rectangle(
    (-0.42, 0), 1.85, 1,  # (x, y), width, height in axes fraction
    linewidth=0.9, edgecolor='black', facecolor='none', transform=ax_bottomright.transAxes, zorder=10,
    clip_on=False
)
ax_bottomright.add_patch(rect)

for t in texts:
    if t.get_text() == 'Deuterated\nchloroform':
        x, y = t.get_position()
        t.set_position((x*1.5, y*0.95))
    if t.get_text() == 'Other\nsolvent':
        x, y = t.get_position()
        t.set_position((x*1.05, y*0.85))
    if t.get_text() == 'Unknown\nsolvent':
        x, y = t.get_position()
        t.set_position((x*1.3, y*0.85))

plots_dir = os.path.join(os.getcwd(), "plots")
fig.savefig(os.path.join(plots_dir, "figure 5.png"), dpi=dpi_for_pub, bbox_inches='tight')

plt.show()

In [ ]:
print(f'Proportion of CHNO: {len(exp_test_df[exp_test_df["CHNO_only"] == True])/len(exp_test_df)}]')
print(f'Proportion of halogens: {len(exp_test_df[exp_test_df["With_halogen"] == True])/len(exp_test_df)}]')
print(f'Proportion of SPB: {len(exp_test_df[exp_test_df["With_SPB"]])/len(exp_test_df)}]')

### figure s8

In [ ]:
# cross-val set proportion of known solvents
exp_train_df['Solvent_Group'] = exp_train_df['Solvent'].apply(label_solvent)


solvent_counts = exp_train_df['Solvent_Group'].value_counts().reindex(
    ['Deuterated\nchloroform', 'Other\nsolvent', 'Unknown\nsolvent'], fill_value=0)
pie_colors = ['#6eda19', '#effe01', '#0dcbb7']

fig, ax = plt.subplots(figsize=(4, 4), dpi=dpi_for_pub)
wedges, texts, autotexts = ax.pie(solvent_counts, labels=solvent_counts.index, colors=pie_colors,
       autopct='%1.1f%%', startangle=90, textprops={'fontsize': 12})
# Add a box around the pie chart
rect = mpatches.Rectangle(
    (-0.2, 0), 1.4, 1,  # (x, y), width, height in axes fraction
    linewidth=0.9, edgecolor='black', facecolor='none', transform=ax.transAxes, zorder=10,
    clip_on=False
)

for t in texts:
    if t.get_text() == 'Deuterated\nchloroform':
        x, y = t.get_position()
        t.set_position((x*1.5, y*0.95))
    if t.get_text() == 'Other\nsolvent':
        x, y = t.get_position()
        t.set_position((x*1.05, y*0.85))
    if t.get_text() == 'Unknown\nsolvent':
        x, y = t.get_position()
        t.set_position((x*1.3, y*0.85))
        
ax.add_patch(rect)
plt.tight_layout()

plots_dir = os.path.join(os.getcwd(), "plots")
plt.savefig(os.path.join(plots_dir, "figure s8.png"), dpi=dpi_for_pub, bbox_inches='tight')

plt.show()

### figure s6

In [ ]:
fig = plt.figure(figsize=(12, 10), dpi=dpi_for_pub)
gs = gridspec.GridSpec(3, 3, wspace=0.3, hspace=0.3)  # 3 rows, 3 columns

axes = [fig.add_subplot(gs[i, j]) for i in range(3) for j in range(3)]

#updated features
new_feat =  (
    '$_{num}$$\\delta$$^{13C}$',
    '$_{max}$$\\delta$$^{13C}$',
    '$_{mean}$$\\delta$$^{13C}$',
    '$_{skew}$$\\delta$$^{13C}$',
    '$_{kurt}$$\\delta$$^{13C}$',
    '$_{num}$$\\delta$$^{1H}$',
    '$_{max}$$\\delta$$^{1H}$',
    '$_{skew}$$\\delta$$^{1H}$'
)

# Left: Feature importances bar chart
colors = ["#e6308a" if "13C" in f else "#89ce00" if "1H" in f else "#888888" for f in new_feat]
axes[0].barh(new_feat, nmrshift_importances, color=[colors[new_feat.index(f)] for f in new_feat])
axes[0].set_xlabel('Importance', fontsize=13)
axes[0].set_ylabel('Feature', fontsize=13)
axes[0].text(-0.15, 1.0, 'A', transform=axes[0].transAxes, fontsize=16, fontweight='bold', va='top', ha='right')

# SHAP scatter plots
dot_size = 3

shap.plots.scatter(nmrshift_explanation[:, "1H Skewness"], ax=axes[1], 
                   color="#89ce00", dot_size=dot_size, show=False)
axes[1].set_ylabel("SHAP Value", labelpad=-1, loc='center')
axes[1].text(0.1, 1.0, 'B', transform=axes[1].transAxes, fontsize=16, fontweight='bold', va='top', ha='right')
axes[1].set_xlabel(r'Z-score normalized $_{skew}$$\delta$$^{1H}$')

shap.plots.scatter(nmrshift_explanation[:, "1H Maximum Shift"], ax=axes[2], 
                   color="#89ce00", dot_size=dot_size, show=False)
axes[2].set_ylabel("SHAP Value", labelpad=-1, loc='center')
axes[2].text(0.1, 1.0, 'C', transform=axes[2].transAxes, fontsize=16, fontweight='bold', va='top', ha='right')
axes[2].set_xlabel(r'Z-score normalized $_{max}$$\delta$$^{1H}$')

shap.plots.scatter(nmrshift_explanation[:, "1H Number of Shifts"], ax=axes[3], 
                   color="#89ce00", dot_size=dot_size, show=False)
axes[3].set_ylabel("SHAP Value", labelpad=-1, loc='center')
axes[3].text(0.1, 1.0, 'D', transform=axes[3].transAxes, fontsize=16, fontweight='bold', va='top', ha='right')
axes[3].set_xlabel(r'Z-score normalized $_{num}$$\delta$$^{1H}$')

shap.plots.scatter(nmrshift_explanation[:, "13C Kurtosis"], ax=axes[4], 
                   color="#e6308a", dot_size=dot_size, show=False)
axes[4].set_ylabel("SHAP Value", labelpad=-1, loc='center')
axes[4].text(0.1, 1.0, 'E', transform=axes[4].transAxes, fontsize=16, fontweight='bold', va='top', ha='right')
axes[4].set_xlabel(r'Z-score normalized $_{kurt}$$\delta$$^{13C}$')

shap.plots.scatter(nmrshift_explanation[:, "13C Skewness"], ax=axes[5], 
                   color="#e6308a", dot_size=dot_size, show=False)
axes[5].set_ylabel("SHAP Value", labelpad=-1, loc='center')
axes[5].text(0.1, 1.0, 'F', transform=axes[5].transAxes, fontsize=16, fontweight='bold', va='top', ha='right')
axes[5].set_xlabel(r'Z-score normalized $_{skew}$$\delta$$^{13C}$')

shap.plots.scatter(nmrshift_explanation[:, "13C Mean Shift"], ax=axes[6], 
                   color="#e6308a", dot_size=dot_size, show=False)
axes[6].set_ylabel("SHAP Value", labelpad=-1, loc='center')
axes[6].text(0.1, 1.0, 'G', transform=axes[6].transAxes, fontsize=16, fontweight='bold', va='top', ha='right')
axes[6].set_xlabel(r'Z-score normalized $_{mean}$$\delta$$^{13C}$')

shap.plots.scatter(nmrshift_explanation[:, "13C Maximum Shift"], ax=axes[7], 
                   color="#e6308a", dot_size=dot_size, show=False)
axes[7].set_ylabel("SHAP Value", labelpad=-1, loc='center')
axes[7].text(0.1, 1.0, 'H', transform=axes[7].transAxes, fontsize=16, fontweight='bold', va='top', ha='right')
axes[7].set_xlabel(r'Z-score normalized $_{max}$$\delta$$^{13C}$')

shap.plots.scatter(nmrshift_explanation[:, "13C Number of Shifts"], ax=axes[8], 
                   color="#e6308a", dot_size=dot_size, show=False)
axes[8].set_ylabel("SHAP Value", labelpad=-1, loc='center')
axes[8].text(0.1, 1.0, 'I', transform=axes[8].transAxes, fontsize=16, fontweight='bold', va='top', ha='right')
axes[8].set_xlabel(r'Z-score normalized $_{num}$$\delta$$^{13C}$')

plt.tight_layout()
plt.subplots_adjust(wspace=0.2)  # Reduce horizontal space between subplots
plt.savefig(os.path.join(plots_dir, "figure s6.png"), dpi=dpi_for_pub, bbox_inches='tight')
plt.show()

### figure s7

In [ ]:
# Plot: Residual vs Molecular Weight
fig, ax = plt.subplots(figsize=(7, 5))
x = exp_test_df['mol_weight']
y = exp_test_df['Residual']
ax.scatter(x, y, s=20, alpha=0.6, edgecolors='black', linewidths=0.4, color='black')
ax.set_xlabel('Molecular Weight (g/mol)')
ax.set_ylabel('ωB97X-D/def2svp computed $\\Delta$E minus experimental NMR-predicted $\\Delta$E (eV)')

# add a trend line
slope, intercept, r_value, p_value, std_err = linregress(x, y)
ax.plot(x, intercept + slope * x, color='red', linestyle='--', linewidth=1, label=f'Linear regression slope={slope:.3g}')
ax.legend()
plt.tight_layout()
plt.subplots_adjust(wspace=0.2)  # Reduce horizontal space between subplots
plt.savefig(os.path.join(plots_dir, "figure s7.png"), dpi=dpi_for_pub, bbox_inches='tight')
plt.show()

### figure s9

In [ ]:
fig = plt.figure(figsize=(10, 6), dpi=dpi_for_pub)
gs = gridspec.GridSpec(1, 2, wspace=0.3)  # 1 rows, 2 columns

axes = [fig.add_subplot(gs[i, j]) for i in range(1) for j in range(2)]

# Plot colored by Temperature (continuous colormap, handle 'Unreported')
x = exp_test_df['Actual'].values
y = exp_test_df['Predicted'].values
temp_col = exp_test_df['Temperature [K]']

# Identify numeric and unreported
is_unreported = temp_col.astype(str).str.lower().str.contains('unreported')

# Convert numeric values, set unreported to NaN
numeric_temp = pd.to_numeric(temp_col.where(~is_unreported), errors='coerce')

# Plot unreported temperatures first
if is_unreported.any():
    axes[0].scatter(x[is_unreported], y[is_unreported], c='#b8cebe', s=18, alpha=0.8, edgecolors='black', linewidths=0.4, label='Unreported Temp', zorder=1)

# Plot reported numeric temperatures
sc = axes[0].scatter(x[~is_unreported], y[~is_unreported], c=numeric_temp[~is_unreported], cmap='viridis', s=18, alpha=1, edgecolors='black', linewidths=0.4, label='Reported Temp', zorder=2)
cbar = plt.colorbar(sc, ax=axes[0])
cbar.set_label('Temperature (K)')

lo = min(x.min(), y.min())
hi = max(x.max(), y.max())
axes[0].plot([lo, hi], [lo, hi], 'r--', linewidth=1, zorder=0)
axes[0].set_xlabel('ωB97X-D/def2svp computed $\\Delta$E (eV)')
axes[0].set_ylabel('Experimental NMR-predicted $\\Delta$E (eV)')
axes[0].legend(title='Temperature Reported')
axes[0].text(-0.1, 1.0, 'A', transform=axes[0].transAxes, fontsize=16, fontweight='bold', va='top', ha='right')

# pie chart

# Create a boolean mask for known (numeric) temperatures
is_numeric_temp = pd.to_numeric(exp_test_df['Temperature [K]'], errors='coerce').notna()
temp_counts = pd.Series(
    [is_numeric_temp.sum(), (~is_numeric_temp).sum()],
    index=['Known\nTemperature', 'Unreported\nTemperature']
)
pie_colors = ['#6eda19', '#0dcbb7']

axes[1].pie(temp_counts, labels=temp_counts.index, colors=pie_colors,
            autopct='%1.1f%%', startangle=90, textprops={'fontsize': 12})
axes[1].text(-0.05, 0.98, 'B', transform=axes[1].transAxes,
             fontsize=16, fontweight='bold', va='top', ha='right')

# Add a box around the pie chart
rect = mpatches.Rectangle(
    (0, 0), 1.0, 1,  # (x, y), width, height in axes fraction
    linewidth=0.9, edgecolor='black', facecolor='none', transform=axes[1].transAxes, zorder=10,
    clip_on=False
)
axes[1].add_patch(rect)

plt.subplots_adjust(wspace=0.1)  # Reduce horizontal space between subplots
plt.savefig(os.path.join(plots_dir, "figure s9.png"), dpi=dpi_for_pub, bbox_inches='tight')
plt.show()